In [ ]:
!nvidia-smi

In [ ]:
import matplotlib.pyplot as plt
x = [1,2,3]
y = [1,1.5,2]
z = [1,1.25,1.5]
v = [1,1.125,1.25]
w = [1,1.0625,1.125]
plt.plot(x,'#1f0a1d')
plt.plot(y,'#334f53')
plt.plot(z,'#45936c')
plt.plot(v,'#9acc77')
plt.plot(w,'#e5ead4')

colors = ['#1f0a1d','#334f53','#45936c','#9acc77','#e5ead4']

# Assets and packages

In [ ]:
!pip install corner
!pip install lenstronomy
!pip install objax

In [ ]:
import gigalens ## CLONE GIGALENS FROM GITHUB/YOUR VERSION

In [ ]:
pip install "jax[gpu]==0.4.33" tensorflow-probability==0.25.0

In [ ]:
import tensorflow_probability.substrates.jax as tfp
import jax
from jax import random
import numpy as np
import optax
from jax import numpy as jnp
import matplotlib as mpl
from matplotlib import pyplot as plt
import seaborn as sns
import optax
from corner import corner

import time
from datetime import timedelta

!pip install lenstronomy
sns.set_style('white')

## Delens functions

In [ ]:
from jax import jit, lax
import functools

''' Functions for delensing'''
@functools.partial(jit)
def _rotate_EPL(x, y, phi):
    cos_phi, sin_phi = jnp.cos(phi), jnp.sin(phi)
    return x * cos_phi + y * sin_phi, -x * sin_phi + y * cos_phi
@functools.partial(jit)
def deriv_EPL(x, y, theta_E, gamma, e1, e2, center_x, center_y):
    phi = jnp.arctan2(e2, e1) / 2
    c = jnp.clip(jnp.sqrt(e1 ** 2 + e2 ** 2), 0, 1)
    q = (1 - c) / (1 + c)
    theta_E_conv = theta_E / (jnp.sqrt((1.0 + q ** 2) / (2.0 * q)))
    b = theta_E_conv * jnp.sqrt((1 + q ** 2) / 2)
    t = gamma - 1

    x, y = x - center_x, y - center_y
    x, y = _rotate_EPL(x, y, phi)

    R = jnp.clip(jnp.sqrt((q * x) ** 2 + y ** 2), 1e-10, 1e10)
    angle = jnp.arctan2(y, q * x)
    f = (1 - q) / (1 + q)
    Cs, Ss = jnp.cos(angle), jnp.sin(angle)
    Cs2, Ss2 = jnp.cos(2 * angle), jnp.sin(2 * angle)
    niter = 18

    def update(n, val):
        prefac = -f * (2 * n - (2 - t)) / (2 * n + (2 - t))
        last_x, last_y, fx, fy = val
        last_x, last_y = prefac * (Cs2 * last_x - Ss2 * last_y), prefac * (
                Ss2 * last_x + Cs2 * last_y
        )
        fx += last_x
        fy += last_y
        return last_x, last_y, fx, fy

    _, _, fx, fy = lax.fori_loop(1, niter, update, (Cs, Ss, Cs, Ss))
    prefac = (2 * b) / (1 + q) * ((b / R) ** (t - 1))
    fx, fy = fx * prefac, fy * prefac
    return _rotate_EPL(fx, fy, -phi)

@functools.partial(jit)
def deriv_shear(x, y, gamma1, gamma2):
  return gamma1 * x + gamma2 * y, gamma2 * x - gamma1 * y

@functools.partial(jit)
def _deriv_EPL_shear(x, y, lens_params):
  x1, y1 = x, y

  f_xi, f_yi = deriv_shear(x, y, **lens_params[0][1])

  f_xi2, f_yi2 = deriv_EPL(x1, y1, **lens_params[0][0])
  f_xi, f_yi = f_xi + f_xi2, f_yi + f_yi2
  return f_xi,f_yi


@functools.partial(jit)
def _beta_EPL_shear(x, y, lens_params):
 x1, y1 = x, y

 f_xi, f_yi = deriv_shear(x, y, **lens_params[0][1])
 beta_x, beta_y = x - f_xi, y - f_yi

 f_xi, f_yi = deriv_EPL(x1, y1, **lens_params[0][0])
 beta_x, beta_y = beta_x - f_xi, beta_y - f_yi

 return beta_x, beta_y

## Magnification functions

In [ ]:
@functools.partial(jit)
def _hessian_differential_square(x, y, kwargs, diff):
        """
        computes the numerical differentials over a finite range for f_xx, f_yy, f_xy from f_x and f_y
        The differentials are computed on the square around (x, y). This minimizes curl.

        :param x: x-coordinate
        :param y: y-coordinate
        :param kwargs: lens model keyword argument list
        :param k: int, list of booleans or None, indicating a subset of lens models to be evaluated
        :param diff: float, scale of the finite differential (diff/2 in each direction used to compute the differential
        :return: f_xx, f_xy, f_yx, f_yy
        # """
        # alpha_ra_pp, alpha_dec_pp = _beta_EPL_shear(x + diff/2, y + diff/2, kwargs)
        # alpha_ra_pn, alpha_dec_pn = _beta_EPL_shear(x + diff/2, y - diff/2, kwargs)

        # alpha_ra_np, alpha_dec_np = _beta_EPL_shear(x - diff / 2, y + diff / 2, kwargs)
        # alpha_ra_nn, alpha_dec_nn = _beta_EPL_shear(x - diff / 2, y - diff / 2, kwargs)

        alpha_ra_pp, alpha_dec_pp = _deriv_EPL_shear(x + diff/2, y + diff/2, kwargs)
        alpha_ra_pn, alpha_dec_pn = _deriv_EPL_shear(x + diff/2, y - diff/2, kwargs)

        alpha_ra_np, alpha_dec_np = _deriv_EPL_shear(x - diff / 2, y + diff / 2, kwargs)
        alpha_ra_nn, alpha_dec_nn = _deriv_EPL_shear(x - diff / 2, y - diff / 2, kwargs)

        f_xx = (alpha_ra_pp - alpha_ra_np + alpha_ra_pn - alpha_ra_nn) / diff / 2
        f_xy = (alpha_ra_pp - alpha_ra_pn + alpha_ra_np - alpha_ra_nn) / diff / 2
        f_yx = (alpha_dec_pp - alpha_dec_np + alpha_dec_pn - alpha_dec_nn) / diff / 2
        f_yy = (alpha_dec_pp - alpha_dec_pn + alpha_dec_np - alpha_dec_nn) / diff / 2

        return f_xx, f_xy, f_yx, f_yy
@functools.partial(jit)
def hessian(x, y, kwargs, diff):
        """
        hessian matrix

        :param x: x-position (preferentially arcsec)
        :type x: numpy array
        :param y: y-position (preferentially arcsec)
        :type y: numpy array
        :param kwargs: list of keyword arguments of lens model parameters matching the lens model classes
        :param k: only evaluate the k-th lens model
        :param diff: float, scale over which the finite numerical differential is computed. If None, then using the
         exact (if available) differentials.
        :param diff_method: string, 'square' or 'cross', indicating whether finite differentials are computed from a
         cross or a square of points around (x, y)
        :return: f_xx, f_xy, f_yx, f_yy components
        """
        return _hessian_differential_square(x, y, kwargs, diff)

@functools.partial(jit)
def hessian_AD(x, y, kwargs): ##This Hessian is returning exactly the same shape as the prev. JAX.jacfwd should have machine number precision, that is, better precision and speed than finite differences
    hessian_fn = jax.jacfwd(lambda x, y: _deriv_EPL_shear(x, y, kwargs), argnums=(0, 1))
    hessian = jnp.array(jax.vmap(hessian_fn, in_axes=(0, 0))(x, y)).reshape(2,2,x.shape[0],-1)
    # print(hessian.shape)
    return hessian[0][0], hessian[0][1], hessian[1][0], hessian[1][1] #f_xx, f_xy, f_yx, f_yy

In [ ]:
@functools.partial(jit)
def magnification(x, y, kwargs, diff=0.0001):
        """
        magnification
        mag = 1/det(A)
        A = 1 - d^2phi/d_ij

        :param x: image plane x-position (preferentially arcsec)
        :type x: numpy array
        :param y: image plane y-position (preferentially arcsec)
        :type y: numpy array
        :param kwargs: list of keyword arguments of lens model parameters matching the lens model classes
        :param k: only evaluate the k-th lens model
        :param diff: float, scale over which the finite numerical differential is computed. If None, then using the
         exact (if available) differentials.
        :param diff_method: string, 'square' or 'cross', indicating whether finite differentials are computed from a
         cross or a square of points around (x, y)
        :return: magnification
        """

        # f_xx, f_xy, f_yx, f_yy = hessian(x, y, kwargs, diff=diff)
        f_xx, f_xy, f_yx, f_yy = hessian_AD(x, y, kwargs)
        det_A = (1 - f_xx) * (1 - f_yy) - f_xy*f_yx
        return det_A #1/det_A

@functools.partial(jit)
def magnification_AD(x, y, kwargs):
        """
        magnification
        mag = 1/det(A)
        A = 1 - d^2phi/d_ij

        :param x: image plane x-position (preferentially arcsec)
        :type x: numpy array
        :param y: image plane y-position (preferentially arcsec)
        :type y: numpy array
        :param kwargs: list of keyword arguments of lens model parameters matching the lens model classes
        :param k: only evaluate the k-th lens model
        :param diff: float, scale over which the finite numerical differential is computed. If None, then using the
         exact (if available) differentials.
        :param diff_method: string, 'square' or 'cross', indicating whether finite differentials are computed from a
         cross or a square of points around (x, y)
        :return: magnification
        """

        f_xx, f_xy, f_yx, f_yy = hessian_AD(x, y, kwargs)
        det_A = (1 - f_xx) * (1 - f_yy) - f_xy*f_yx
        return det_A #1/det_A

## Time Delay functions

In [ ]:
@functools.partial(jit)
def _lensing_pot_EPL(x, y, theta_E, gamma, e1, e2, center_x, center_y):
    """

    :param x: x-coordinate in image plane
    :param y: y-coordinate in image plane
    :param theta_E: Einstein radius
    :param gamma: power law slope
    :param e1: eccentricity component
    :param e2: eccentricity component
    :param center_x: profile center
    :param center_y: profile center
    :return: lensing potential
    """
    alpha_x, alpha_y = deriv_EPL(x, y, theta_E, gamma, e1, e2, center_x, center_y)
    t = gamma - 1
    x_ = x - center_x
    y_ = y - center_y
    f_ = (x_ * alpha_x + y_ * alpha_y) / (2 - t)
    return f_


@functools.partial(jit)
def _lensing_pot_shear(x, y, gamma1, gamma2, ra_0=0, dec_0=0): #Check ra_0 and dec_0 influence
    """

    :param x: x-coordinate (angle)
    :param y: y0-coordinate (angle)
    :param gamma1: shear component
    :param gamma2: shear component
    :param ra_0: x/ra position where shear deflection is 0
    :param dec_0: y/dec position where shear deflection is 0
    :return: lensing potential
    """
    x_ = x - ra_0
    y_ = y - dec_0
    f_ = 1 / 2.0 * (gamma1 * x_ * x_ + 2 * gamma2 * x_ * y_ - gamma1 * y_ * y_)
    return f_


@functools.partial(jit)
def _lensing_pot_EPL_shear(x, y, lens_params):
 x1, y1 = x, y

 f_1 = _lensing_pot_shear(x, y, **lens_params[0][1])

 f_2 = _lensing_pot_EPL(x1, y1, **lens_params[0][0])

 return f_1 + f_2

In [ ]:
@functools.partial(jit)
def fermat_potential(x, y, kwargs_lens, sourcePos_x, sourcePos_y): #This function coincides with that of Lenstronomy to the 7th decimal place
  lensing_potential = _lensing_pot_EPL_shear(x, y, kwargs_lens)
  geometry = ((x - sourcePos_x) ** 2 + (y - sourcePos_y) ** 2) / 2.0 #There are not Line of Sight corrections
  return geometry - lensing_potential

In [ ]:
##BREAK DOWN OF THE COSMOLOGY FOR FLATLAMBDA CDM

# def comoving_distance_z1z2(z1,z2, H0):  #Coincides with Astropy values and units
#     return (c / H0)* (1 /1000) * cosmo._integral_comoving_distance_z1z2_scalar(z1, z2) #Mpc

# def comoving_distance(z, H0):  #Coincides with Astropy values and units
#     return comoving_distance_z1z2(0,z, H0) #Mpc

# def angular_diameter_distance_z1z2(z1,z2, H0): #Coincides with Astropy values and units
#     return (comoving_distance(z2, H0) - comoving_distance(z1, H0))/(1+z2) #Mpc

# # @functools.partial(jit)
# def time_delay(x, y, kwargs_lens, sourcePos_x, sourcePos_y, z_lens, z_source, H0):
#     fermat_pot = fermat_potential(x, y, kwargs_lens, sourcePos_x, sourcePos_y)

#     dd = angular_diameter_distance_z1z2(0, z_lens, H0)
#     ds = angular_diameter_distance_z1z2(0, z_source, H0)
#     dds = angular_diameter_distance_z1z2(z_lens, z_source, H0)

#     ddt = (1 + z_lens) * dd * ds / dds * 3.0856775800000002e+22 #Assume kappa_ext = 0 #3.0856775800000002e+22 Mpc

#     return ddt / c * fermat_pot / 86400 * (4.84813681109536e-06)**2 #86400 sec in a day. #4.84813681109536e-06 arcsec to rad

In [ ]:
##JIT COMPILABLE FUNCTION THAT DOES NOT CALCULATE THE SAME COMOVING DISTANCE EACH ITERATION
@functools.partial(jit)
def time_delay(x, y, kwargs_lens, sourcePos_x, sourcePos_y, z_lens, z_source): #Agrees with astropy with a 2e-5% relative error
    fermat_pot = fermat_potential(x, y, kwargs_lens, sourcePos_x, sourcePos_y)
    H0 = kwargs_lens[0][3]['H0']
    dd = (2.9979246e8 / H0)* (1 /1000) * int_comoving_dist_lens / (1+z_lens)
    ds = (2.9979246e8 / H0)* (1 /1000) * int_comoving_dist_source / (1+z_source)
    dds = ((2.9979246e8 / H0)* (1 /1000) * int_comoving_dist_source - (2.9979246e8 / H0)* (1 /1000) * int_comoving_dist_lens) / (1+z_source)

    ddt = (1 + z_lens) * dd * ds / dds * 3.0856775800000002e+22 #Assume kappa_ext = 0 #3.0856775800000002e+22 Mpc

    return ddt / 2.9979246e8 * fermat_pot / 86400 * (4.84813681109536e-06)**2 #2.9979246e8 is c in m/s #86400 sec in a day. #4.84813681109536e-06 arcsec to rad

## Point source light class

In [ ]:
import gigalens.profile


class Sersic(gigalens.profile.LightProfile):
    _name = "SERSIC"
    _params = ["R_sersic", "n_sersic", "center_x", "center_y"]

    @functools.partial(jit, static_argnums=(0,))
    def light(self, x, y, R_sersic, n_sersic, center_x, center_y, Ie=None):
        R = self.distance(x, y, center_x, center_y)
        bn = 1.9992 * n_sersic - 0.3271
        ret = jnp.exp(-bn * ((R / R_sersic) ** (1 / n_sersic) - 1.0))
        return ret[jnp.newaxis, ...] if self.use_lstsq else (Ie * ret)

    @functools.partial(jit, static_argnums=(0,))
    def distance(self, x, y, cx, cy, e1=None, e2=None):
        if e1 is None:
            e1 = jnp.zeros_like(cx)
        if e2 is None:
            e2 = jnp.zeros_like(cx)
        phi = jnp.arctan2(e2, e1) / 2
        c = jnp.sqrt(e1 ** 2 + e2 ** 2)
        q = (1 - c) / (1 + c)
        dx, dy = x - cx, y - cy
        cos_phi, sin_phi = jnp.cos(phi), jnp.sin(phi)
        xt1 = (cos_phi * dx + sin_phi * dy) * jnp.sqrt(q)
        xt2 = (-sin_phi * dx + cos_phi * dy) / jnp.sqrt(q)
        return jnp.sqrt(xt1 ** 2 + xt2 ** 2)


class SersicEllipse(Sersic):
    _name = "SERSIC_ELLIPSE"
    _params = ["R_sersic", "n_sersic", "e1", "e2", "center_x", "center_y"]

    @functools.partial(jit, static_argnums=(0,))
    def light(self, x, y, R_sersic, n_sersic, e1, e2, center_x, center_y, Ie=None):
        R = self.distance(x, y, center_x, center_y, e1, e2)
        bn = 1.9992 * n_sersic - 0.3271
        ret = Ie*jnp.exp(-bn * ((R / R_sersic) ** (1 / n_sersic) - 1.0))
        return ret[jnp.newaxis, ...] if self.use_lstsq else (Ie * ret)


class CoreSersic(Sersic):
    _name = "CORE_SERSIC"
    _params = [
        "R_sersic",
        "n_sersic",
        "Rb",
        "alpha",
        "gamma",
        "e1",
        "e2",
        "center_x",
        "center_y",
    ]

    @functools.partial(jit, static_argnums=(0,))
    def light(
            self,
            x,
            y,
            R_sersic,
            n_sersic,
            Rb,
            alpha,
            gamma,
            e1,
            e2,
            center_x,
            center_y,
            Ie=None,
    ):
        R = self.distance(x, y, center_x, center_y, e1, e2)
        bn = 1.9992 * n_sersic - 0.3271
        ret = ((1 + (Rb / R) ** alpha) ** (gamma / alpha) * jnp.exp(-bn * (
                (R ** alpha + Rb ** alpha)
                / R_sersic ** alpha ** 1.0
                / (alpha * n_sersic)
        ) - 1.0))
        return ret[jnp.newaxis, ...] if self.use_lstsq else (Ie * ret)


class Point(Sersic):
    _name = "POINT"
    _params = ["center_x", "center_y"]
    '''
    simulate a gaussian, return brigtest pixel
    I have not kept bn which was to ensure Ie is the high light brigtness.
    Insteady I want Ie to be the brightest flux.
    I use R_scale to basically make the light fall to zero quickly.
    I dont have a lstsq fit version of this yet.  See return statement for Sersic above.
    '''
    @functools.partial(jit, static_argnums=(0,))
    def light(self, x, y, center_x, center_y, Ie=None, R_sersic=None, n_sersic=None):
        R = self.distance(x, y, center_x, center_y)
#         R_scale = 0.002
#         pix = Ie * jnp.exp(-(R / R_scale) ** 2 )
#         maxpix = jnp.max(pix)
#         ret = jnp.array(maxpix)
#         # return ret[jnp.newaxis, ...] if self.use_lstsq else (Ie * ret)
        n_fixed = 1000
        # R_fixed = 0.002
        R_fixed = 0.0015
        b_fixed = 1.9992 * n_fixed - 0.3271
        ret = Ie*jnp.exp(-b_fixed * ((R / R_fixed) ** (1 / n_fixed) - 1.0))
        return ret[jnp.newaxis, ...] if self.use_lstsq else (Ie * ret)
        # return ret[jnp.newaxis, ...]

## Bright points class

In [ ]:
from lenstronomy.Data.pixel_grid import PixelGrid
import tensorflow as tf
class BrightestPoints:
    def __init__(self, number_of_images = 4, num_pixels=750, grid_size=250, delta_pix=0.0006, supersample=1):
        self.number_of_images = number_of_images
        self.num_pixels = num_pixels
        self.grid_size = int(grid_size)
        self.delta_pix = delta_pix
        self.supersample = supersample
    def find_brightest_points(self, img):
        image = tf.reshape(img, [1, self.num_pixels, self.num_pixels, 1])
        patches = tf.image.extract_patches(
            images=image,
            sizes=[1, self.grid_size, self.grid_size, 1],
            strides=[1, self.grid_size, self.grid_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )
        patches = tf.reshape(
            patches,
            [int(self.num_pixels / self.grid_size), int(self.num_pixels / self.grid_size), self.grid_size, self.grid_size, 1]
        )
        max_values = tf.reduce_max(patches, axis=(2, 3))
        result_grid = tf.squeeze(max_values)
        flat_grid = tf.reshape(result_grid, [-1])
        values, indices = tf.math.top_k(flat_grid, k= self.number_of_images)
        brightest_points = tf.stack([
            tf.squeeze(tf.where(img == values[i]), 0) for i in range(self.number_of_images)
        ])
        return brightest_points
    def pix_to_arcsec(self, brightest_points):
        """Convert pixel coordinates to arcsecond coordinates.
        Args:
            brightest_points (tf.Tensor): Tensor containing the pixel coordinates of the brightest points.
                                          Shape: (number_of_images, 2)
            num_pix (int, optional): Number of pixels in the image. Defaults to 750.
            delta_pix (float, optional): Pixel scale in arcseconds. Defaults to 0.0006.
            supersample (int, optional): Supersampling factor. Defaults to 1.
        Returns:
            tf.Tensor: Tensor containing the converted arcsecond coordinates of the brightest points.
                      Shape: (number_of_images, 2)
        """
        lo = np.arange(0, self.supersample * self.num_pixels, dtype=np.float32)
        lo = np.min(lo - np.mean(lo))
        transform_pix2angle = (tf.eye(2) * self.delta_pix) / self.supersample
        ra_at_xy_0, dec_at_xy_0 = np.squeeze((transform_pix2angle @ ([[lo], [lo]])))
        kwargs_pixel_rot = {
            "nx": self.supersample * self.num_pixels,
            "ny": self.supersample * self.num_pixels,  # number of pixels per axis
            "ra_at_xy_0": ra_at_xy_0,  # RA at pixel (0,0)
            "dec_at_xy_0": dec_at_xy_0,  # DEC at pixel (0,0) # bottom-left corner
            "transform_pix2angle": np.array(transform_pix2angle),
        }
        pixel_grid_rot = PixelGrid(**kwargs_pixel_rot)
        img_x, img_y = (
            pixel_grid_rot._x_grid.astype(np.float32),
            pixel_grid_rot._y_grid.astype(np.float32),
        )
        img_x, img_y = tf.expand_dims(img_x, axis=0), tf.expand_dims(img_y, axis=0)
        img_x, img_y = tf.repeat(img_x, repeats = self.number_of_images, axis = 0), tf.repeat(img_y, repeats = self.number_of_images, axis = 0)
        column = tf.range(tf.shape(brightest_points)[0])
        reshaped_column = tf.reshape(column, (-1, 1))

        reshaped_column = tf.cast(reshaped_column, dtype=tf.int64)
        brightest_points = tf.cast(brightest_points, dtype=tf.int64)
        enumerated_brightest_points = tf.concat([reshaped_column, brightest_points], axis=1)

        grid_indices, x_indices, y_indices = tf.unstack(enumerated_brightest_points, axis=-1)
        x_arcsec = tf.gather_nd(img_x, tf.stack([grid_indices, x_indices, y_indices], axis=-1))
        y_arcsec = tf.gather_nd(img_y, tf.stack([grid_indices, x_indices, y_indices], axis=-1))
        return jnp.array(x_arcsec, dtype=jnp.float32), jnp.array(y_arcsec, dtype=jnp.float32) #Convert them into JAX friendly

## Modeling functions

### MAP

In [ ]:
import functools

import jax.random
import optax
# import tensorflow_probability.substrates.jax as tfp
import time
from jax import jit, pmap
from jax import numpy as jnp
from tensorflow_probability.substrates.jax import (
    distributions as tfd,
    bijectors as tfb,
    experimental as tfe,
)
from tqdm.auto import trange

import gigalens.inference
import gigalens.jax.simulator as sim
import gigalens.model

def MAP(
        optimizer: optax.GradientTransformation,
        start=None,
        n_samples=500,
        num_steps=350,
        seed=0,
):
    dev_cnt = jax.device_count()
    n_samples = (n_samples // dev_cnt) * dev_cnt
    # lens_sim = sim.LensSimulator(
    #     self.phys_model,
    #     self.sim_config,
    #     bs=n_samples // dev_cnt,
    # )
    seed = jax.random.PRNGKey(seed)

    start = (
        prob_model.prior.sample(n_samples, seed=seed)
        if start is None
        else start
    )
    params = jnp.stack(prob_model.bij.inverse(start)).T

    opt_state = optimizer.init(params)

#     def loss(z):
#         lp, chisq = self.prob_model.log_prob(lens_sim, z)
#         return -jnp.mean(lp) / jnp.size(self.prob_model.observed_image), chisq

    def loss(z):
        # return - point_loss(z)
        return -jnp.squeeze(point_loss(z)) #NEW jnp squeeze. All it does is convert n_GPUs, n_samples//n_GPUs , 1 into n_GPUs, n_samples//n_GPUs
        #It works because of vmap. Value and grad needs a scalar function.



    # loss_and_grad = jax.pmap(jax.value_and_grad(loss, has_aux=False))
    loss_and_grad = jax.pmap(jax.vmap(jax.value_and_grad(loss), in_axes=(0))) #NEW vmap

    def update(params, opt_state):
        splt_params = jnp.array(jnp.split(params, dev_cnt, axis=0))
        # (_, chisq), grads = loss_and_grad(splt_params)
        loss_value, grads = loss_and_grad(splt_params)
        grads = jnp.concatenate(grads, axis=0)
        # chisq = jnp.concatenate(chisq, axis=0)
        loss_value = jnp.concatenate(loss_value, axis=0)

        updates, opt_state = optimizer.update(grads, opt_state)
        new_params = optax.apply_updates(params, updates)
        return loss_value, new_params, opt_state

    with trange(num_steps) as pbar:
        for _ in pbar:
            loss, params, opt_state = update(params, opt_state)
            pbar.set_description(
                f"Chi-squared: {float(jnp.nanmin(loss, keepdims=True)[0]):.3f}"
            )
    return params

### SVI

In [ ]:
import tqdm
from tqdm.auto import trange

import multiprocessing
import time

def SVI(
        start,
        optimizer: optax.GradientTransformation,
        n_vi=250,
        init_scales=1e-3,
        num_steps=500,
        seed=0,
):
    dev_cnt = jax.device_count()
    seeds = jax.random.split(jax.random.PRNGKey(seed), dev_cnt)
    n_vi = (n_vi // dev_cnt) * dev_cnt
    # lens_sim = sim.LensSimulator(
    #     self.phys_model,
    #     self.sim_config,
    #     bs=n_vi // dev_cnt,
    # )
    scale = (
        jnp.diag(jnp.ones(jnp.size(start))) * init_scales
        if jnp.size(init_scales) == 1
        else init_scales
    )
    cov_bij = tfp.bijectors.FillScaleTriL(diag_bijector=tfb.Exp(), diag_shift=1e-6)
    qz_params = jnp.concatenate(
        [jnp.squeeze(start), cov_bij.inverse(scale)], axis=0
    )
    replicated_params = jax.tree_map(lambda x: jnp.array([x] * dev_cnt), qz_params)

    n_params = jnp.size(start)

    def elbo(qz_params, seed):
        mean = qz_params[:n_params]
        cov = cov_bij.forward(qz_params[n_params:])
        qz = tfd.MultivariateNormalTriL(loc=mean, scale_tril=cov)
        z = qz.sample(n_vi // dev_cnt, seed=seed)
        lps = qz.log_prob(z)
        return jnp.mean(lps - point_loss(z))

    elbo_and_grad = jit(jax.value_and_grad(jit(elbo), argnums=(0,)))

    @functools.partial(pmap, axis_name="num_devices")
    def get_update(qz_params, seed):
        val, grad = elbo_and_grad(qz_params, seed)
        return jax.lax.pmean(val, axis_name="num_devices"), jax.lax.pmean(
            grad, axis_name="num_devices"
        )

    opt_state = optimizer.init(replicated_params)
    loss_hist = []
    with trange(num_steps) as pbar:
        for step in pbar:
            loss, (grads,) = get_update(replicated_params, seeds)
            loss = float(jnp.mean(loss))
            seeds = jax.random.split(seeds[0], dev_cnt)
            updates, opt_state = optimizer.update(grads, opt_state)
            replicated_params = optax.apply_updates(replicated_params, updates)
            pbar.set_description(f"ELBO: {loss:.3f}")
            loss_hist.append(loss)
    mean = replicated_params[0, :n_params]
    cov = cov_bij.forward(replicated_params[0, n_params:])
    qz = tfd.MultivariateNormalTriL(loc=mean, scale_tril=cov)
    return qz, loss_hist

### HMC

In [ ]:
def HMC(
        q_z,
        init_eps=0.3,
        init_l=3,
        n_hmc=50,
        num_burnin_steps=250,
        num_results=750,
        max_leapfrog_steps=30,
        num_steps_between_results = 0,
        seed=0,
):
    dev_cnt = jax.device_count()
    seeds = jax.random.split(jax.random.PRNGKey(seed), dev_cnt)
    n_hmc = (n_hmc // dev_cnt) * dev_cnt
    # lens_sim = sim.LensSimulator(
    #     self.phys_model,
    #     self.sim_config,
    #     bs=n_hmc // dev_cnt,
    # )
    momentum_distribution = tfd.MultivariateNormalFullCovariance(
        loc=jnp.zeros_like(q_z.mean()),
        covariance_matrix=jnp.linalg.inv(q_z.covariance()),
    )

    # @jit
    # def log_prob(z):
    #     return self.prob_model.log_prob(lens_sim, z)[0]

    @pmap
    def run_chain(seed):
        start = q_z.sample(n_hmc // dev_cnt, seed=seed)
        num_adaptation_steps = int(num_burnin_steps * 0.8)
        mc_kernel = tfe.mcmc.PreconditionedHamiltonianMonteCarlo(
            # target_log_prob_fn=log_prob,
            target_log_prob_fn=lambda param: point_loss(param),
            momentum_distribution=momentum_distribution,
            step_size=init_eps,
            num_leapfrog_steps=init_l,
        )

        mc_kernel = tfe.mcmc.GradientBasedTrajectoryLengthAdaptation(
            mc_kernel,
            num_adaptation_steps=num_adaptation_steps,
            max_leapfrog_steps=max_leapfrog_steps,
        )
        mc_kernel = tfp.mcmc.DualAveragingStepSizeAdaptation(
            inner_kernel=mc_kernel, num_adaptation_steps=num_adaptation_steps,
            target_accept_prob=0.75, #DEFAULT: 0.75
            reduce_fn=tfp.math.reduce_logmeanexp #DEFAULT: NOT USED
        )

        return tfp.mcmc.sample_chain(
            num_results=num_results,
            num_burnin_steps=num_burnin_steps,
            num_steps_between_results=num_steps_between_results, #DEFAULT: 0
            current_state=start,
            trace_fn=lambda _, pkr: None,
            seed=seed,
            kernel=mc_kernel,
        )

    start = time.time()
    ret = run_chain(seeds)
    end = time.time()
    print(f"Sampling took {(end - start):.1f}s")
    return ret

# Simulation

In [ ]:
from tqdm.auto import trange, tqdm

In [ ]:
# from gigalens.jax.inference import ModellingSequence
from gigalens.jax.model import ForwardProbModel, BackwardProbModel
from gigalens.model import PhysicalModel
from gigalens.jax.simulator import LensSimulator
from gigalens.simulator import SimulatorConfig
from gigalens.jax.profiles.light import sersic
from gigalens.jax.profiles.mass import epl, shear

## Set truth

In [ ]:
systems = [
    [
        {'theta_E': 1.167, 'gamma': 2., 'e1': 0.049, 'e2': 0.083, 'center_x': 0.005, 'center_y': 0.011},  #Mass distribution model
        {'gamma1': 0.078, 'gamma2': 0.015},  #External shear
        {'amp':15},
        {'H0':70},
    ],
    [
        {'theta_E': 1.158, 'gamma': 2.113, 'e1': 0.029, 'e2': -0.123, 'center_x': -0.010, 'center_y': 0.010},
        {'gamma1': 0.018, 'gamma2': 0.025},
        {'amp':15},
        {'H0':70},
    ],
     [
        {'theta_E': 1.360, 'gamma': 1.75, 'e1': 0.019, 'e2': -0.132, 'center_x': 0.010, 'center_y': -0.012},
        {'gamma1': 0.038, 'gamma2': 0.015},
        {'amp':15},
        {'H0':70},
      ],
     [
        {'theta_E': 1.350, 'gamma': 1.9, 'e1': 0.029, 'e2': 0.053, 'center_x': -0.02, 'center_y': 0.025},
        {'gamma1': 0.012, 'gamma2': -0.025},
        {'amp':15},
        {'H0':70},
      ],
    [
        {'theta_E': 1.050, 'gamma': 2.3, 'e1': 0.029, 'e2': 0.053, 'center_x': 0.015, 'center_y': 0.010},
        {'gamma1': 0.078, 'gamma2': 0.015},
        {'amp':15},
        {'H0':70},
    ],
     [
        {'theta_E': 1.025, 'gamma': 2.35, 'e1': -0.329, 'e2': -0.353, 'center_x': -0.001, 'center_y': 0.00},
        {'gamma1': 0.013, 'gamma2': 0.035},
        {'amp':15},
        {'H0':70},
      ],
    [
        {'theta_E': 1.025, 'gamma': 2.1, 'e1': 0.1, 'e2': -0.05, 'center_x': -0.001, 'center_y': 0.00},  #Mass distribution model
        {'gamma1': 0.013, 'gamma2': 0.035},  #External shear
        {'amp':1},
        {'H0':70},
    ],
    [
        {'theta_E': 2.15, 'gamma': 2.1, 'e1': 0.1, 'e2': -0.05, 'center_x': -0.001, 'center_y': 0.00},  #Mass distribution model
        {'gamma1': 0.013, 'gamma2': 0.035},  #External shear
        {'amp':1},
        {'H0':70},
    ],     
 ]

source_truth = [
    [
        {'center_x': 0.004, 'center_y': 0.009, 'Ie': 2815.973},
    ],
    [
        {'center_x': 0.085, 'center_y': -0.080, 'Ie': 2815.973},
    ],
     [
        {'center_x': 0.145, 'center_y': 0.105, 'Ie': 2815.973},
      ],
     [
        {'center_x': 0.050, 'center_y': 0.024, 'Ie': 2815.973},
      ],
    [
        {'center_x': 0.145, 'center_y': 0.125, 'Ie': 2815.973},
    ],
    [
        {'center_x': 0.045, 'center_y': -0.040, 'Ie': 2815.973},
    ],
    [
        {'center_x': 0.005, 'center_y': -0.040, 'Ie': 2815.973},
    ],
    [
        {'center_x': 0.005, 'center_y': -0.040, 'Ie': 2815.973},
    ],
 ]

num_systems = len(systems)

## Simulate

In [ ]:
phys_model = PhysicalModel(
    [
        epl.EPL(50), #Mass distribution model
        shear.Shear(), #External shear
    ],
    [
        sersic.SersicEllipse(), #Lens light
    ],
    [
        Point(), #Source light
    ]
)

In [ ]:
delta_pix = 0.008
num_pix = 750
# delta_pix = 0.025
# num_pix = 40

supersample = 1
kernel = np.load('.npy').astype(np.float32) #PSF

sim_config = SimulatorConfig(delta_pix=delta_pix, num_pix=num_pix, supersample=supersample,
                             kernel = kernel,
                            ) #Delta pix set in 0.02 instead of 0.065

lens_sim = LensSimulator(phys_model, sim_config, bs=1)

In [ ]:
from lenstronomy.LensModel.lens_model import LensModel
from lenstronomy.Plots import lens_plot

In [ ]:
lensed_images = []
fig, axes = plt.subplots(num_systems, 4, figsize = (20, 5*num_systems))
lens_model_list = ['EPL', 'SHEAR']
lensModel = LensModel(lens_model_list=lens_model_list)

for i in range(num_systems):
  truth = [
      systems[i], #Mass model
      [
          {'R_sersic': 0.350, 'n_sersic': 2.586, 'e1': -0.262, 'e2': 0.239, 'center_x': 0.011, 'center_y': 0.048, 'Ie': 35.055}
       ], # Lens Light
      source_truth[i],
      ]

  sim_img = lens_sim.simulate(truth)
  lens_omitted_img= lens_sim.simulate([truth[0],[],truth[2]]) # Omit lens light
  source_img = lens_sim.simulate([[],[],truth[2]]) # Source
  lensed_images.append(lens_omitted_img)
  fig0 = axes[i,0].imshow(sim_img, norm=mpl.colors.PowerNorm(0.5, vmin=0, vmax=8),)
  axes[i,0].set_title(f'Simulated supernova {i}')
  fig1 = axes[i,1].imshow(lens_omitted_img, norm=mpl.colors.PowerNorm(0.5, vmin=0, vmax=8),)
  axes[i,1].set_title('Lens light omitted')
  fig2 = axes[i,2].imshow(source_img, norm=mpl.colors.PowerNorm(0.5, vmin=0, vmax=8),)
  axes[i,2].set_title('Unlensed supernova simulation')

  kwargs_spep = truth[0][0]
  kwargs_shear = truth[0][1]
  kwargs_lens = [kwargs_spep, kwargs_shear]

  sourcePos_x = truth[2][0]['center_x']
  sourcePos_y = truth[2][0]['center_y']

  converter = BrightestPoints(number_of_images = 4, num_pixels=num_pix, grid_size=num_pix/3, delta_pix=delta_pix, supersample=1)
  brightest_pixels = converter.find_brightest_points(lensed_images[i])
  x_arcsec, y_arcsec = converter.pix_to_arcsec(brightest_pixels)

  lens_plot.lens_model_plot(axes[i,3], lensModel=lensModel,
                            kwargs_lens=kwargs_lens,numPix=num_pix, deltaPix=delta_pix,
                            sourcePos_x=sourcePos_x, sourcePos_y=sourcePos_y, point_source=False,
                            with_caustics=True, fast_caustic=True,

                            coord_inverse=False)
  axes[i,3].plot(x_arcsec,y_arcsec,'bd')
  axes[i,3].plot(sourcePos_x,sourcePos_y,'y*')

  axes[i,3].invert_yaxis()


  # cax = plt.axes([0.95, 0.3, 0.02, 0.2])  # [left, bottom, width, height]
  # colorbar = fig.colorbar(fig0, ax = axes, cax = cax, extend = 'max',)#anchor = (-0.3,0.5))
  # colorbar.outline.set_edgecolor('grey')
  # colorbar.outline.set_linewidth(0.5)


plt.setp(axes[:,:3], xticks=[], yticks=[])
plt.show()

# Determine the brightest points

In [ ]:
from lenstronomy.LensModel.lens_model import LensModel
from lenstronomy.Plots import lens_plot
from lenstronomy.Data.imaging_data import ImageData

In [ ]:
index = 7 #What system to analyze

truth = [
      systems[index], #Mass model
      [
          {'R_sersic': 0.350, 'n_sersic': 2.586, 'e1': -0.262, 'e2': 0.239, 'center_x': 0.011, 'center_y': 0.048, 'Ie': 35.055}
       ], # Lens Light
      source_truth[index]# Source Light
      ]

kwargs_spep = truth[0][0]
kwargs_shear = truth[0][1]
kwargs_lens = [kwargs_spep, kwargs_shear]

sourcePos_x = truth[2][0]['center_x']
sourcePos_y = truth[2][0]['center_y']



delta_pix_2 = 0.0006*2 ## More accuracy for determining the brightest points. For real data we could fit a gaussian and find its centroid (usually more accurate)
num_pix_2 = 5000


sim_config_2 = SimulatorConfig(delta_pix=delta_pix_2, num_pix=num_pix_2, supersample=supersample,
                             # kernel = kernel,
                            ) #Delta pix set in 0.02 instead of 0.065

lens_sim_2 = LensSimulator(phys_model, sim_config_2, bs=1)

sim_img_trial = lens_sim_2.simulate(truth)


converter = BrightestPoints(number_of_images = 4, num_pixels=num_pix_2, grid_size=num_pix_2/2, delta_pix=delta_pix_2, supersample=1)
brightest_pixels = converter.find_brightest_points(sim_img_trial)
x_arcsec, y_arcsec = converter.pix_to_arcsec(brightest_pixels)

print(x_arcsec)
print(y_arcsec)


extent = (-num_pix_2/2*delta_pix_2, num_pix_2/2*delta_pix_2, num_pix_2/2*delta_pix_2, -num_pix_2/2*delta_pix_2)
fig,ax = plt.subplots(1,1,figsize = (10,8))
plt.imshow(sim_img_trial, norm=mpl.colors.PowerNorm(0.5, vmin=0, vmax=8),extent=extent,)

kwargs_caustics = {
        "color_crit": (0,0,0,0),
          "color_caustic": 'green'}

kwargs_criticals = {
          "color_crit": 'r',
          "color_caustic": (0,0,0,0)}

kwargs_both = {
        "color_crit": 'red',
          "color_caustic": 'green'}


lens_plot.lens_model_plot(
ax,
lensModel,
kwargs_lens,
numPix=num_pix_2,
deltaPix=delta_pix_2,
point_source=False,
with_caustics=True,
with_convergence=False,
coord_inverse=False,
fast_caustic=True,
kwargs_caustics = kwargs_both,
)

plt.plot(x_arcsec,y_arcsec,'cd', ms = 10, alpha = 0.8, label = 'Images')
plt.plot(sourcePos_x,sourcePos_y,'y*', ms = 15, alpha = 0.8, label = 'Source')
plt.grid(False)
plt.xticks([])
plt.yticks([])
plt.tight_layout()
plt.gca().invert_yaxis()
plt.legend()
plt.show()

In [ ]:
#Delensing
delensed_positions = _beta_EPL_shear(x_arcsec, y_arcsec, truth)
plt.plot(delensed_positions[0], delensed_positions[1], marker = 'D',markersize = 5,linestyle = '', color = 'navy', label = 'Delensed positions of the images')
plt.title('Source plane')
plt.xlabel('x - position (arcsec)')
plt.ylabel('y - position (arcsec)')

center_x_truth, center_y_truth = truth[2][0]['center_x'], truth[2][0]['center_y']
plt.plot(center_x_truth, center_y_truth, 'b*', label = 'Source Truth Position', color = 'black')

#Relative errors
error_x = np.abs(np.array(delensed_positions[0]) - center_x_truth)*100/center_x_truth
error_y = np.abs(np.array(delensed_positions[1]) - center_y_truth)*100/center_y_truth
print('Relative errors of x estimation after delens (%):',error_x)
print('Relative errors of y estimation after delens (%):',error_y)



plt.xlim(center_x_truth*(1 - 0.1*np.mean(error_x)), center_x_truth*(1 + 0.1*np.mean(error_x)))
plt.ylim(center_y_truth*(1 - 0.1*np.mean(error_y)), center_y_truth*(1 + 0.1*np.mean(error_y)))
plt.annotate(
   '(' + str(np.round(center_x_truth,4)) + ','+ str(np.round(center_y_truth,4)) + ')',
   xy=(center_x_truth,center_y_truth),
   xytext=(40, 30),
   textcoords='offset points', ha='right', va='bottom',
  #  bbox=dict(boxstyle='round,pad=0.5', fc='darkgrey'),
   arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0', color = 'darkgrey', )
   )

plt.legend(bbox_to_anchor=(1, 1))
plt.show()

# Eccentricities: e1 and e2 approach

## Loss function

### **Time Delay** + Flux + Distance Information

In [ ]:
weight_dist = 3.*1e3

weight_flux = 5.*1e9

weight_time_delay = 3.*1e3



def point_loss(params, test = False):
    # params is in unconstrained space
    # print(params.shape)
    params = list(params.T) #NEW
    const_param = prob_model.bij.forward(params) #Constrained space


    #MINIMIZE COMPACTNESS OF SOURCE PLANE
    tens_param = jnp.array(_beta_EPL_shear(x, y, const_param))
    source_param = jnp.mean(tens_param, axis = 1)
    sourcePos_x_param, sourcePos_y_param = source_param[0], source_param[1]
    shifted_tens_param = jnp.concatenate([tens_param[:,-1:,:], tens_param[:,:-1,:]], axis=1)
    dist_loss = jnp.mean(jnp.sum((tens_param - shifted_tens_param)**2, axis = 0)**0.5, axis = 0)


    #MINIMIZE FLUX DIFFERENCE
    mag_param = magnification(x,y,const_param)
    init_amp = const_param[0][2]['amp']
    flux_param = mag_param**2/init_amp**2  #Actually the inverse of flux
    flux_loss = jnp.mean((flux_param - flux_truth)**2, axis = 0)


    #MINIMIZE TIME DELAY DIFFERENCE
    time_delay_param = time_delay(x, y, const_param, sourcePos_x_param, sourcePos_y_param, z_lens, z_source)
    time_delay_param -= time_delay_param[0] #Relative fermatPot #x and y must be sorted such that x[0] is the earliest to arrive
    time_delay_loss = jnp.mean((time_delay_param - time_delay_truth)**2, axis = 0)

    if test:
      return - dist_loss*weight_dist, - flux_loss*weight_flux, - time_delay_loss*weight_time_delay, prior.log_prob(const_param), prob_model.bij.forward_log_det_jacobian(params)
    else:
      return - dist_loss*weight_dist - flux_loss*weight_flux - time_delay_loss*weight_time_delay + prior.log_prob(const_param)+prob_model.bij.forward_log_det_jacobian(params)

## Modeling

### Initialization

#### Set prior

In [ ]:
lens_prior = tfd.JointDistributionSequential( #UNIFORM PRIOR
  [tfd.JointDistributionNamed(
      dict(
          # theta_E=tfd.Uniform(low = 0.0, high = 0.5),
          # theta_E=tfd.Uniform(low = 0.5, high = 2.0),
          theta_E=tfd.Uniform(low = 1.5, high = 3.0), #3rd system H0
          #  gamma=tfd.Uniform(1.3, 2.8,),
          # gamma=tfd.Normal(2, 0.25),
          gamma=tfd.TruncatedNormal(2.0, 0.25, 1.5, 2.5),
          #  e1=tfd.Uniform(-0.1, 0.1),
          #  e2=tfd.Uniform(-0.1, 0.1),
          #  e1=tfd.TruncatedNormal(0, 0.1, -0.2, 0.2),
          #  e2=tfd.TruncatedNormal(0, 0.1, -0.2, 0.2),
          e1=tfd.Normal(0, 0.1),
          e2=tfd.Normal(0, 0.1),
          center_x=tfd.Normal(0, 0.1),
          center_y=tfd.Normal(0, 0.1),
          #  center_x=tfd.TruncatedNormal(0, 0.1, -0.2, 0.2),
          #  center_y=tfd.TruncatedNormal(0, 0.1, -0.2, 0.2),
          #  center_x=tfd.Uniform(-0.1, 0.1),
          #  center_y=tfd.Uniform(-0.1, 0.1),
          )
      ),
   tfd.JointDistributionNamed(
       dict(
          #  gamma1=tfd.TruncatedNormal(0, 0.1, -0.2, 0.2),
          #  gamma2=tfd.TruncatedNormal(0, 0.1, -0.2, 0.2),
           gamma1=tfd.Normal(0, 0.1),
           gamma2=tfd.Normal(0, 0.1),
          # gamma1=tfd.Uniform(-0.1, 0.1),
          # gamma2=tfd.Uniform(-0.1, 0.1),
          ),
       ),
   tfd.JointDistributionNamed( ####TYPE IA SNe HAVE UNC ~0.1-0.2 MAGNITUDES, WHICH CORRESPONDS TO 10-20% BRIGHTNESS UNC
       dict(
           amp=tfd.Normal(1, 0.1*1), #(10% sigma)
          ),
       ),
   tfd.JointDistributionNamed( ##H0 prior
       dict(
           H0=tfd.Uniform(0, 150),
          ),
       ),
  ]
  )


prior = tfd.JointDistributionSequential(
    [lens_prior,]
)


prob_model = ForwardProbModel(prior, sim_img, background_rms=0.2, exp_time=100)

#### Format brightest points and use measurements

In [ ]:
from astropy.cosmology import FlatLambdaCDM
use_measurements = False

if use_measurements:
    #Plug in your measurements

    x_arcsec_measured = jnp.array([0.05, -0.14999999, 0.17, -0.06999999], dtype = jnp.float32)
    y_arcsec_measured = jnp.array([-0.14999999, -0.10999999, 0.03, 0.13], dtype = jnp.float32)
    
    flux_measured = jnp.array([17.78,12.22,9.04,4.13], dtype = jnp.float32)
    
    time_delay_measured = jnp.array([7.3,4.4,3.0,0.], dtype = jnp.float32)


    z_lens, z_source = 0.2262, 0.3544 #SN Zwicky, e.g.
    delta_pix = 0.001
    num_pix = 750
    sim_config = SimulatorConfig(delta_pix=delta_pix, num_pix=num_pix, supersample=supersample,
                                 kernel = kernel,)
    lens_sim = LensSimulator(phys_model, sim_config, bs=1)



    #Eg, FLAT LAMBDA CDM
    cosmo = FlatLambdaCDM(H0=70, Om0=0.3, Ob0=0.) #The value for H0 here is irrelevant because it is only used for comoving distance (which we have implemented independently)

    int_comoving_dist_lens = cosmo._integral_comoving_distance_z1z2_scalar(0, z_lens)
    int_comoving_dist_source = cosmo._integral_comoving_distance_z1z2_scalar(0, z_source)

    sort_order = jnp.argsort(time_delay_measured,-1)
    time_delay_measured = jnp.take(time_delay_measured, sort_order)
    flux_measured = jnp.take(flux_measured, sort_order)
    x_arcsec_measured = jnp.take(x_arcsec_measured, sort_order)
    y_arcsec_measured = jnp.take(y_arcsec_measured, sort_order)

    chains = 1
    x = jnp.repeat(x_arcsec_measured[..., jnp.newaxis], chains, axis=-1)
    y = jnp.repeat(y_arcsec_measured[..., jnp.newaxis], chains, axis=-1)

    print('Time delay measured', time_delay_measured)
    print(x,y)

else:
    z_lens, z_source = 0.2262, 0.3544 #SN Zwicky, e.g.

    #Eg, FLAT LAMBDA CDM
    cosmo = FlatLambdaCDM(H0=70, Om0=0.3, Ob0=0.) #The value for H0 here is irrelevant because it is only used for comoving distance (which we have implemented independently

    int_comoving_dist_lens = cosmo._integral_comoving_distance_z1z2_scalar(0, z_lens)
    int_comoving_dist_source = cosmo._integral_comoving_distance_z1z2_scalar(0, z_source)

    #Sorting the brightest points as the order of later arrival
    delens_sorting = _beta_EPL_shear(x_arcsec, y_arcsec, truth)
    sourcePos_x_truth_sorting = jnp.mean(delens_sorting[0])
    sourcePos_y_truth_sorting = jnp.mean(delens_sorting[1])

    fermatPot_sorting = fermat_potential(x_arcsec, y_arcsec, truth, sourcePos_x_truth_sorting, sourcePos_y_truth_sorting)

    sort_order = jnp.argsort(fermatPot_sorting,-1)
    x_arcsec_sorted = jnp.take(x_arcsec, sort_order)
    y_arcsec_sorted = jnp.take(y_arcsec, sort_order)

    #Check it is sorted
    fermatPot_sorted = fermat_potential(x_arcsec_sorted, y_arcsec_sorted, truth, sourcePos_x_truth_sorting, sourcePos_y_truth_sorting)
    fermatPot_sorted -= fermatPot_sorted[0]


    #Use x and y ordered as the time delay
    bs = 1
    x = jnp.repeat(x_arcsec_sorted[..., jnp.newaxis], bs, axis=-1)
    y = jnp.repeat(y_arcsec_sorted[..., jnp.newaxis], bs, axis=-1)

    print('True x', x_arcsec_sorted)
    print('True y', y_arcsec_sorted)

    sim_time_delay_sorted = time_delay(x_arcsec_sorted, y_arcsec_sorted, truth, sourcePos_x_truth_sorting, sourcePos_y_truth_sorting, z_lens, z_source)
    sim_time_delay_sorted -= sim_time_delay_sorted[0]
    print('Time delay simulated and sorted', sim_time_delay_sorted)

    print('True init amp =', truth[0][2]['amp'])
    print('True source position', sourcePos_x_truth_sorting,sourcePos_y_truth_sorting)

In [ ]:
if use_measurements:
    #INPUT MEASUREMENTS
    print('USING MEASUREMENTS')
    flux_truth = 1/jnp.array(flux_measured, )**2
    flux_truth = jnp.repeat(flux_truth[..., jnp.newaxis], chains, axis=-1)

    fermatPot_truth = jnp.array(fermatPot_measured, ) #Already relative to the minimum as measured
    fermatPot_truth = jnp.repeat(fermatPot_truth[..., jnp.newaxis], chains, axis=-1)


    print('Inverse flux',flux_truth)
    print('FP truth',fermatPot_truth)
else:
    #SIMULATE TRUE SOURCE
    delens_truth = _beta_EPL_shear(x, y, truth)
    tens_truth = jnp.array(delens_truth)
    source_pos_truth = jnp.mean(tens_truth, axis = 1)
    sourcePos_x_truth = source_pos_truth[0]
    sourcePos_y_truth = source_pos_truth[1]
    #SIMULATE TRUE FLUX
    mag_truth = magnification(x,y,truth)
    init_amp = truth[0][2]['amp']
    flux_truth = mag_truth**2/init_amp**2  #Actually the inverse of flux squared
    print('True magnifications', 1/mag_truth)

    #SIMULATE TRUE FERMAT POTENTIAL
    fermatPot_truth = fermat_potential(x,y,truth, sourcePos_x_truth, sourcePos_y_truth)
    fermatPot_truth -= fermatPot_truth[0] #Measured fermat pot (Obtained by measuring time delay and changing units)
    time_delay_truth = time_delay(x, y, truth, sourcePos_x_truth, sourcePos_y_truth, z_lens, z_source)
    time_delay_truth -= time_delay_truth[0]
    print('True flux', 1/jnp.sqrt(flux_truth))
    print('True TD', time_delay_truth)
    ##Magnification of lenstronomy is not quite the same as in TF pipeline

### MAP

In [ ]:
# n_map = 500 #Original
n_map = 1000
n_steps = 400*2
# schedule_fn = tf.keras.optimizers.schedules.PolynomialDecay(1e-2, 300, 1e-2/5) #Original

schedule_fn = optax.polynomial_schedule(init_value=-1e-1, end_value=-1e-2/5,
                                      power=0.5, transition_steps=n_steps)
optimizer = optax.chain(
  optax.scale_by_adam(),
  optax.scale_by_schedule(schedule_fn),
)


start_MAP = time.time()


# MAP_sample = MAP(optimizer=optimizer, n_samples=n_map, num_steps=300, seed=0) #Original
MAP_sample = MAP(optimizer=optimizer, n_samples=n_map, num_steps=n_steps, seed=0)


end_MAP = time.time()

elapsed_MAP = end_MAP-start_MAP

print('MAP takes ',timedelta(seconds=elapsed_MAP))

In [ ]:
lps = point_loss(MAP_sample)
best = MAP_sample[jnp.nanargmax(lps)]

In [ ]:

print(truth[0])

In [ ]:
prob_model.bij.forward(list(best))

In [ ]:
point_loss(jnp.array(best), test = True)

Plot graphs for the obtained MAP - truth.

In [ ]:
if use_measurements:
    delens_modeled_MAP = _beta_EPL_shear(x_arcsec_measured, y_arcsec_measured, [prob_model.bij.forward(list(best))[0],[],[]])
    tens_modeled_MAP = jnp.array(delens_modeled_MAP)
    source_modeled_MAP = jnp.mean(tens_modeled_MAP, axis = 1)
    sourcePos_x_modeled_MAP = source_modeled_MAP[0]
    sourcePos_y_modeled_MAP = source_modeled_MAP[1]
    modeled_truth_MAP = [
      prob_model.bij.forward(list(best))[0], #Mass model
      [

       ], # Lens Light
             [
        {'center_x': sourcePos_x_modeled_MAP, 'center_y': sourcePos_y_modeled_MAP, 'Ie': 2*281.973},
    ],
      ]

    plt.plot(x_arcsec_measured,y_arcsec_measured,'cd', ms = 15, alpha = 0.8, label = 'True positions')

else:

    modeled_truth_MAP = [
      prob_model.bij.forward(list(best))[0], #Mass model
      [

       ], # Lens Light
      source_truth[index]# Source Light
      ]
    delens_modeled_MAP = _beta_EPL_shear(x_arcsec_sorted, y_arcsec_sorted, modeled_truth_MAP)
    sourcePos_x_modeled_MAP = jnp.mean(delens_modeled_MAP[0])
    sourcePos_y_modeled_MAP = jnp.mean(delens_modeled_MAP[1])
    plt.plot(x_arcsec,y_arcsec,'cd', ms = 15, alpha = 0.8, label = 'True positions')

modeled_img_MAP = lens_sim.simulate(modeled_truth_MAP)

extent = (-num_pix/2*delta_pix, num_pix/2*delta_pix, num_pix/2*delta_pix, -num_pix/2*delta_pix)
plt.imshow(modeled_img_MAP, norm=mpl.colors.PowerNorm(0.5, vmin=0, vmax=8),extent=extent,)
plt.grid(False)
plt.gca().invert_yaxis()
# plt.xticks([])
# plt.yticks([])
plt.tight_layout()
plt.title('Modeled image MAP')
plt.legend()
plt.show()

### SVI

In [ ]:
# schedule_fn = optax.polynomial_schedule(init_value=-1e-6, end_value=-3e-3, #Learns less
#                                       power=2, transition_steps=1000) #JAX demo

# schedule_fn = optax.polynomial_schedule(init_value=-0.0, end_value=-4e-2, #Learns more
#                                       power=1.0, transition_steps=500) #Original from tf point source

schedule_fn = optax.polynomial_schedule(init_value=-1e-6, end_value=-3e-3,
                                      power=2.0, transition_steps=2500)

optimizer = optax.chain(
  optax.scale_by_adam(),
  optax.scale_by_schedule(schedule_fn),
)


start_SVI = time.time()

# q_z, losses = SVI(optimizer=optimizer, start=best, n_vi=500, num_steps=600) #Original from tf point source
# q_z, losses = SVI(optimizer=optimizer, start=best, n_vi=1000, num_steps=1500) #JAX demo
q_z, losses = SVI(optimizer=optimizer, start=best, n_vi=2000, num_steps=3000)
plt.plot(losses)


end_SVI = time.time()

elapsed_SVI = end_SVI-start_SVI

print('SVI takes ',timedelta(seconds=elapsed_SVI))

In [ ]:
def is_pos_def(x):
    return np.all(np.linalg.eigvals(x) >= 0)

is_pos_def(np.array(q_z.covariance()))

In [ ]:
truth[0]

In [ ]:
prob_model.bij.forward(list(q_z.mean()))

### HMC

In [ ]:
import pickle

restart = True


if restart:
    num_burnin_steps = 250
    # num_burnin_steps = 500
    num_results = 1000
    # num_results = 3000
    # n_hmc = 12
    n_hmc = 24
    # n_hmc = 48
    # num_steps_between_results = 0
    num_steps_between_results = 20
    # num_steps_between_results = 50
    start_HMC = time.time()

    # samples = HMC(q_z=q_z, n_hmc=50, init_eps=0.5, init_l=3, max_leapfrog_steps=100,
    #                                    num_burnin_steps=250, num_results = 750, num_steps_between_results = 0) #Original

    samples = HMC(q_z=q_z, n_hmc=n_hmc, init_eps=0.5, init_l=3, max_leapfrog_steps=100,
                                       num_burnin_steps=num_burnin_steps, num_results = num_results, num_steps_between_results = num_steps_between_results)


    end_HMC = time.time()

    elapsed_HMC = end_HMC-start_HMC

    elapsed_modeling = elapsed_MAP + elapsed_SVI + elapsed_HMC

    # with open(f'', 'wb') as f:
    # pickle.dump((samples,elapsed_HMC,elapsed_modeling), f)

else:
    date = '' ##SAVE YOUR MODEL

    with open(f'' + date + '.pickle', 'rb') as f:
        samples, elapsed_HMC, elapsed_modeling = pickle.load(f)


print('HMC takes ',timedelta(seconds=elapsed_HMC))

#### Rhat and Chains

In [ ]:
n_params = 10 ##Fitting for H0 too!
# n_params = 9
num_results = jnp.transpose(samples.all_states, (1,2,0,3)).shape[0]
n_hmc = jnp.transpose(samples.all_states, (1,2,0,3)).reshape(num_results, -1, n_params).shape[1]

In [ ]:
smp = jnp.transpose(samples.all_states, (1,2,0,3)).reshape((-1, n_params))

rhat= tfp.mcmc.potential_scale_reduction(jnp.transpose(samples.all_states, (1,2,0,3)), independent_chain_ndims=2)


sample_chains = jnp.transpose(samples.all_states, (1,2,0,3)).reshape(num_results, -1, n_params)

rhat_dict = prob_model.pack_bij.forward(list(rhat))
print(rhat_dict)


print('rhat:')
for i in range(len(rhat_dict)):
    if i==0:
        print('mass params:')
    elif i==1:
        print('lens light:')
    elif i==2:
        print('source light:')
    for j in range(len(rhat_dict[i])):
        for key, val in rhat_dict[i][j].items():
            if val > 1.5 and val <2.0:
                print('\x1b[0;33;48m' + '{:>10} {:6.3f}'.format(key, val) + '\x1b[0m', end="")
            elif val >= 2.0:
                print('\x1b[0;31;48m' + '{:>10} {:6.3f}'.format(key, val) + '\x1b[0m', end="")
            else:
                print('{:>10} {:6.3f}'.format(key, val), end="")
        print('')
    print('')

worst_idx = np.argmax(rhat)
print('worse rhat:', rhat[worst_idx])

ess =tfp.mcmc.effective_sample_size(jnp.transpose(samples.all_states, (1,2,0,3)).reshape(-1, n_hmc, n_params),cross_chain_dims=1,) #ESS properly combining different chains, ie, ESS BETWEEN chains. cross_chain_dims=1 where 1 refers to the second index of samples, that is, the number of chains. This tells which argument is to be interpreted as the chains
ess_chains =tfp.mcmc.effective_sample_size(jnp.transpose(samples.all_states, (1,2,0,3)).reshape(-1, n_hmc, n_params),filter_beyond_positive_pairs=True) #ESS per chain, ie, ESS WITHIN chain: (n_chains, n_params).
ess_chains_avg = np.mean(ess_chains, axis = 0) #Computes the average of ESS per chain across all chains

print('Max and min ESS between chains', ess.max(), ess.min()) ##"The total ESS should be at least 400", Vehtari+ 2021 (TF doc)
print('Max and min avg ESS within chains', ess_chains_avg.max(), ess_chains_avg.min()) ##"Each of the split chains should have an average ESS estimate of at least 50", Vehtari+ 2021 (TF doc)

ess_dict = prob_model.pack_bij.forward(list(ess))
ess_chains_avg_dict = prob_model.pack_bij.forward(list(ess_chains_avg.T))

print('ESS between chains:',ess_dict)
print('Avg ESS within chains:', ess_chains_avg_dict)

print('ess:')
for i in range(len(ess_dict)):
    if i==0:
        print('mass params:')
    elif i==1:
        print('lens light:')
    elif i==2:
        print('source light:')
    for j in range(len(ess_dict[i])):
        for key, val in ess_dict[i][j].items():
            if rhat_dict[i][j][key] > 1.5 and rhat_dict[i][j][key] <2.0:
                print('\x1b[0;33;48m' + '{:>10} {:6.3g}'.format(key, val) + '\x1b[0m', end="")
            elif rhat_dict[i][j][key] >= 2.0:
                print('\x1b[0;31;48m' + '{:>10} {:6.3g}'.format(key, val) + '\x1b[0m', end="")
            else:
                print('{:>10} {:6.3g}'.format(key, val), end="")
        print('')
    print('')


best_bf_bij = jnp.median(smp, axis=0)

best_params_hmc = prob_model.bij.forward(list(best_bf_bij))


smp_physical = prob_model.bij.forward(list(smp.T))


mass_params = smp_physical[0]

In [ ]:
get_samples = lambda x: jnp.array([
    x[0][0]['theta_E'],
    x[0][0]['gamma'],
    x[0][0]['e1'],
    x[0][0]['e2'],
    x[0][0]['center_x'],
    x[0][0]['center_y'],
    x[0][1]['gamma1'],
    x[0][1]['gamma2'],
    x[0][2]['amp'],
    x[0][3]['H0'],
])


physical_samples = get_samples(prob_model.bij.forward(list(jnp.transpose(sample_chains,(2,0,1)))))
# physical_samples = get_samples(prob_model.pack_bij.forward(list(jnp.transpose(sample_chains,(2,0,1)))))

from scipy.stats import norm

colors = [
    '#1f0a1d',   # Dark reddish-purple
    '#8b4513',   # Saddle brown
    '#b22222',    # Firebrick
    '#4b0082',   # Indigo
    '#800000',   # Maroon
    '#6a5acd',   # Slate blue
    '#2e8b57',   # Sea green
    '#006400',   # Dark green
    '#556b2f',   # Dark olive green
    'navy',      # Dark blue
]




fig = plt.figure(figsize=(10, 3))
gs = plt.GridSpec(1, 2, width_ratios=[1, .4], height_ratios=[1])
ax1 = plt.subplot(gs[0, 0],)

ax1.grid(True)


start_chain = 0
num_chains = 10
for i in range(start_chain,start_chain+num_chains):
  ax1.plot(physical_samples[0,:,i],
          colors[i-start_chain],
          # label = f'{i}',
           alpha = 0.6)
  plt.title('HMC $\\theta_E$')
  plt.xlabel('Iterations')
#  plt.legend(title = 'Chains', loc = 'center left', bbox_to_anchor=(1.1, 0.5))

ax2 = plt.subplot(gs[0, 1], sharey=ax1)


for i in range(start_chain,start_chain+num_chains):
  prob, bins = np.histogram(physical_samples[0,:,i], density = True, bins = 40)
  (mu, sigma) = norm.fit(physical_samples[0,:,i])

  pdf = norm.pdf(bins, mu, sigma)
  ax2.plot(pdf, bins,
           colors[i-start_chain],
           linewidth=2, alpha = 0.8)

ax2.grid(True)
plt.gca().axes.get_xaxis().set_visible(False)
plt.tick_params(axis='y', labelsize=0)
plt.gca().axes.get_yaxis().set_visible(True)
plt.tight_layout()
plt.show()


fig = plt.figure(figsize=(10, 3))
gs = plt.GridSpec(1, 2, width_ratios=[1, .4], height_ratios=[1])
ax1 = plt.subplot(gs[0, 0],)

ax1.grid(True)



for i in range(start_chain,start_chain+num_chains):
  ax1.plot(physical_samples[1,:,i],
          colors[i-start_chain],
          # label = f'{i}',
           alpha = 0.6)
  plt.title('HMC $\\gamma$')
  plt.xlabel('Iterations')
#  plt.legend(title = 'Chains', loc = 'center left', bbox_to_anchor=(1.1, 0.5))

ax2 = plt.subplot(gs[0, 1], sharey=ax1)


for i in range(start_chain,start_chain+num_chains):
  prob, bins = np.histogram(physical_samples[1,:,i], density = True, bins = 40)
  (mu, sigma) = norm.fit(physical_samples[1,:,i])

  pdf = norm.pdf(bins, mu, sigma)
  ax2.plot(pdf, bins,
           colors[i-start_chain],
           linewidth=2, alpha = 0.8)

ax2.grid(True)
plt.gca().axes.get_xaxis().set_visible(False)
plt.tick_params(axis='y', labelsize=0)
plt.gca().axes.get_yaxis().set_visible(True)
plt.tight_layout()
plt.show()



fig = plt.figure(figsize=(10, 3))
gs = plt.GridSpec(1, 2, width_ratios=[1, .4], height_ratios=[1])
ax1 = plt.subplot(gs[0, 0],)

ax1.grid(True)



for i in range(start_chain,start_chain+num_chains):
  ax1.plot(physical_samples[10,:,i],
          colors[i-start_chain],
          # label = f'{i}',
           alpha = 0.6)
  plt.title('HMC $H_0$')
  plt.xlabel('Iterations')
#  plt.legend(title = 'Chains', loc = 'center left', bbox_to_anchor=(1.1, 0.5))

ax2 = plt.subplot(gs[0, 1], sharey=ax1)


for i in range(start_chain,start_chain+num_chains):
  prob, bins = np.histogram(physical_samples[10,:,i], density = True, bins = 40)
  (mu, sigma) = norm.fit(physical_samples[10,:,i])

  pdf = norm.pdf(bins, mu, sigma)
  ax2.plot(pdf, bins,
           colors[i-start_chain],
           linewidth=2, alpha = 0.8)

ax2.grid(True)
plt.gca().axes.get_xaxis().set_visible(False)
plt.tick_params(axis='y', labelsize=0)
plt.gca().axes.get_yaxis().set_visible(True)
plt.tight_layout()
plt.show()

In [ ]:
# with open(f'', 'wb') as f:
#     pickle.dump((samples,elapsed_HMC,elapsed_modeling), f)

#### Cornerplot and Rhats

In [ ]:
get_samples = lambda x: jnp.array([
    x[0][0]['theta_E'],
    x[0][0]['gamma'],
    x[0][0]['e1'],
    x[0][0]['e2'],
    x[0][0]['center_x'],
    x[0][0]['center_y'],
    x[0][1]['gamma1'],
    x[0][1]['gamma2'],
    x[0][2]['amp'],
    x[0][3]['H0'],
])


test_tot = np.column_stack((mass_params[0]['theta_E'],
                            mass_params[0]['gamma'],
                            mass_params[0]['e1'],
                            mass_params[0]['e2'],
                            mass_params[0]['center_x'],
                            mass_params[0]['center_y'],
                            mass_params[1]['gamma1'],
                            mass_params[1]['gamma2'],
                            mass_params[2]['amp'],
                            mass_params[3]['H0'],
                           ))

markers = get_samples(truth)
if use_measurements:
    mass_corner_fig = corner(test_tot, labels=[r"$\theta_E$", r"$\gamma$", r"$\epsilon_1$", r"$\epsilon_2$", r"$x_{lens}$", r"$y_{lens}$",
                                r"$\gamma_1$", r"$\gamma_2$", r'$Amp$', r'$H_0$'],
                    title_fmt = '.3f',
                    show_titles=True,
                    plot_contours = True, #[0.5,1,1.5,2]*sigma containing 11.8%, 39.3%, 67.5% and 86.4% of the samples (2D sigma levels)
                    fill_contours=True,
                    # smooth = 0.5,
                    contourf_kwargs = {"colors": [(0.0, 0.0, 0.5, 0.0),(0.0, 0.0, 0.5, 0.4), (0.0, 0.0, 0.5, 0.6), (0.0, 0.0, 0.5, 0.8), (0.0, 0.0, 0.5, 1.0)]},
                    label_kwargs = {'fontsize': 19}, title_kwargs = {'fontsize': 19},);
else:
     mass_corner_fig = corner(test_tot, labels=[r"$\theta_E$", r"$\gamma$", r"$\epsilon_1$", r"$\epsilon_2$", r"$x_{lens}$", r"$y_{lens}$",
                                r"$\gamma_1$", r"$\gamma_2$", r'$Amp$', r'$H_0$'],
                    title_fmt = '.3f',
                    truths = markers,
                    truth_color = '#006400',
                    show_titles=True,
                    plot_contours = True, #[0.5,1,1.5,2]*sigma containing 11.8%, 39.3%, 67.5% and 86.4% of the samples (2D sigma levels)
                    fill_contours=True,
                    # smooth = 0.5,
                    contourf_kwargs = {"colors": [(0.0, 0.0, 0.5, 0.0),(0.0, 0.0, 0.5, 0.4), (0.0, 0.0, 0.5, 0.6), (0.0, 0.0, 0.5, 0.8), (0.0, 0.0, 0.5, 1.0)]},
                    label_kwargs = {'fontsize': 19}, title_kwargs = {'fontsize': 19},);


axes = mass_corner_fig.get_axes()

# Set the fontsize of the ticks
fontsize = 15  # Adjust the fontsize as per your preference
for ax in axes:
    ax.tick_params(axis='both', which='major', labelsize=fontsize)

plt.show()

In [ ]:
# best_HMC = prob_model.pack_bij.forward([np.median(prob_model.pack_bij.inverse(prob_model.bij.forward(samples)).reshape(-1,n_params), axis = 0)])

best_HMC = best_params_hmc

#### Predictions

In [ ]:
if use_measurements:
    delens_modeled = _beta_EPL_shear(x_arcsec_measured, y_arcsec_measured, [best_HMC[0],[],[]])
    tens_modeled = jnp.array(delens_modeled)
    source_modeled = jnp.mean(tens_modeled, axis = 1)
    sourcePos_x_modeled = source_modeled[0]
    sourcePos_y_modeled = source_modeled[1]
    modeled_truth = [
      best_HMC[0], #Mass model
      [

       ], # Lens Light
             [
        {'center_x': sourcePos_x_modeled, 'center_y': sourcePos_y_modeled, 'Ie': 2*280.973},
    ],
      ]

else:

    modeled_truth = [
      best_HMC[0], #Mass model
      [

       ], # Lens Light
      source_truth[index]# Source Light
      ]
    delens_modeled = _beta_EPL_shear(x_arcsec_sorted, y_arcsec_sorted, modeled_truth)
    sourcePos_x_modeled = jnp.mean(delens_modeled[0])
    sourcePos_y_modeled = jnp.mean(delens_modeled[1])

modeled_img = lens_sim.simulate(modeled_truth)

extent = (-num_pix/2*delta_pix, num_pix/2*delta_pix, num_pix/2*delta_pix, -num_pix/2*delta_pix)
plt.imshow(modeled_img,
           norm=mpl.colors.PowerNorm(0.5, vmin=0, vmax=8),
           extent=extent,
          )
plt.colorbar()
plt.grid(False)
plt.gca().invert_xaxis()
# plt.xticks([])
# plt.yticks([])
plt.tight_layout()
plt.title('Modeled image')
plt.show()

In [ ]:
truth[0]

In [ ]:
best_HMC[0]

In [ ]:
physical_samples_dict = prob_model.bij.forward(list(smp.T))
#x,y definitions depend on use_measurements = true/false (see def above)



modeled_FP_samples = fermat_potential(x,y,[physical_samples_dict[0],[],[]],sourcePos_x_modeled,sourcePos_y_modeled)
modeled_TD_samples = time_delay(x, y, [physical_samples_dict[0],[],[]], sourcePos_x_modeled, sourcePos_y_modeled, z_lens, z_source)
modeled_TD_samples -= modeled_TD_samples[0]

TD_unc = np.std(modeled_TD_samples, axis = 1)


### std(amp/detA) = amp_min/detA_min * sqrt((std(amp)/amp_min)**2 + (std(detA)/detA_min)**2)
detA_samples = magnification(x,y,[physical_samples_dict[0],[],[]])
detA_std = np.std(detA_samples, axis = 1)
amp_samples = physical_samples_dict[0][2]['amp']
amp_std = np.std(amp_samples,)

print('Check this uncertainty is small for Taylor Approx',detA_std, amp_std)
detA_min = (magnification(x,y,modeled_truth)).T
amp_min = modeled_truth[0][2]['amp']
flux_unc = np.abs(amp_min/detA_min * np.sqrt((amp_std/amp_min)**2 + (detA_std/detA_min)**2))[0]

print('Error propagation',flux_unc)


modeled_flux_samples = physical_samples_dict[0][2]['amp']/magnification(x,y,[physical_samples_dict[0],[],[]])
flux_unc_2 = np.std(modeled_flux_samples, axis = 1)
print('Std of 1/samples',flux_unc_2)
print('Final unc',flux_unc_2)

In [ ]:
print('Total Modeling time ',timedelta(seconds=elapsed_modeling))



converter = BrightestPoints(number_of_images = 4, num_pixels=num_pix, grid_size=num_pix/2, delta_pix=delta_pix, supersample=1)
brightest_pixels_modeled = converter.find_brightest_points(modeled_img)
x_arcsec_modeled, y_arcsec_modeled = converter.pix_to_arcsec(brightest_pixels_modeled)


extent = (-num_pix/2*delta_pix, num_pix/2*delta_pix, num_pix/2*delta_pix, -num_pix/2*delta_pix)
fig,ax = plt.subplots(1,1,figsize = (10,8))

lens_model_list = ['EPL', 'SHEAR']
lensModel = LensModel(lens_model_list=lens_model_list)


# Calculate positions for the compass rose in data coordinates
compass_x_center = extent[1] - 0.2 * (extent[1] - extent[0])
compass_y_center = extent[2] + 0.2 * (extent[3] - extent[2])

# Define compass directions and offsets
directions = ['N', 'E']
positions = [
    (compass_x_center, compass_y_center),
    (compass_x_center, compass_y_center),
]
offsets = [
    (0, 0.08 * (extent[2] - extent[3])),
    (0.08 * (extent[1] - extent[0]), 0),
]

if use_measurements:
    best_HMC_np = np.median(physical_samples.reshape((n_params,-1)).T, axis = 0)
    # best_HMC_np = np.array(best_bf_bij)
    lens_center_x = modeled_truth[0][0]['center_x']
    lens_center_y = modeled_truth[0][0]['center_y']
    angle = 0.*np.pi/180
    x_arcsec_modeled, y_arcsec_modeled = ((x_arcsec_modeled-lens_center_x)*jnp.cos(angle) - (y_arcsec_modeled-lens_center_y)*jnp.sin(angle) + lens_center_x,(x_arcsec_modeled-lens_center_x)*jnp.sin(angle) + (y_arcsec_modeled-lens_center_y)*jnp.cos(angle) + lens_center_y)
    kwargs_spep = {'theta_E': best_HMC_np[0], 'gamma': best_HMC_np[1], 'e1': best_HMC_np[2], 'e2': best_HMC_np[3], 'center_x': best_HMC_np[4], 'center_y': best_HMC_np[5]}  #Mass distribution model
    kwargs_shear = {'gamma1': best_HMC_np[6], 'gamma2': best_HMC_np[7]}
    kwargs_lens = [kwargs_spep, kwargs_shear]


    plt.imshow(modeled_img,
               norm=mpl.colors.PowerNorm(0.5, vmin=0, vmax=8),
               extent=extent,)

    lens_plot.lens_model_plot(
    ax,
    lensModel,
    kwargs_lens,
    numPix=num_pix,
    deltaPix=delta_pix,
    point_source=False,
    with_caustics=True,
    with_convergence=False,
    coord_inverse=False,
    fast_caustic=True,
    kwargs_caustics = kwargs_both,
    )

    modeled_TD = time_delay(x_arcsec_modeled, y_arcsec_modeled, modeled_truth, sourcePos_x_modeled, sourcePos_y_modeled, z_lens, z_source)
    sort_order_modeled = jnp.argsort(modeled_TD,-1)
    modeled_TD = jnp.take(modeled_TD, sort_order_modeled)
    modeled_TD -= modeled_TD[0]

    modeled_flux = modeled_truth[0][2]['amp']/magnification(x_arcsec_modeled,y_arcsec_modeled,modeled_truth)
    modeled_flux = jnp.take(modeled_flux, sort_order_modeled)





    for i in np.arange(len(x_arcsec_measured)):
        annotation_str = (
        # f"{i+1}\n"
        f"mod_TD: {modeled_TD[i]:.3f}±{TD_unc[i]:.3f}\n"
        f"obs_TD: {time_delay_measured[i]:.3f}\n"
        f"mod_flux: {modeled_flux[i][0]:.3f}±{flux_unc[i]:.3f}\n"
        f"obs_flux: {flux_measured[i]:.3f}"
        )

        plt.annotate(
        annotation_str,
        (x_arcsec_measured[i] - np.sign(x_arcsec_measured[i])*0.0 -0.0, y_arcsec_measured[i] - 0.0),
        fontsize=10,
        color='white'
        )

        letter_image = ['A','B','D','C'] #ORIGINAL
        plt.annotate(
        letter_image[i],
        # (x_arcsec_measured[i] - 0.06, y_arcsec_measured[i] - 0.02),
        (x_arcsec_measured[i]-0.01, y_arcsec_measured[i] + 0.03),
        fontsize=20,
        color='white'
        )

        plt.plot(x_arcsec_measured[i],y_arcsec_measured[i],'kd', ms = 15, alpha = 0.8, markerfacecolor='none')

    plt.plot(x_arcsec_measured,y_arcsec_measured,'d', ms = 15, alpha = 0.8, label = 'Observed position',color = 'lightblue')
    plt.plot(sourcePos_x_modeled,sourcePos_y_modeled,'*', ms = 10, alpha = 0.8, label = 'Source Position', color = 'yellow')

else:
    best_HMC_np = np.median(physical_samples.reshape((n_params,-1)).T, axis = 0)
    lens_center_x = modeled_truth[0][0]['center_x']
    lens_center_y = modeled_truth[0][0]['center_y']
    # angle = 0.*np.pi/180
    # x_arcsec_modeled, y_arcsec_modeled = ((x_arcsec_modeled-lens_center_x)*jnp.cos(angle) - (y_arcsec_modeled-lens_center_y)*jnp.sin(angle) + lens_center_x,(x_arcsec_modeled-lens_center_x)*jnp.sin(angle) + (y_arcsec_modeled-lens_center_y)*jnp.cos(angle) + lens_center_y)
    kwargs_spep = {'theta_E': best_HMC_np[0], 'gamma': best_HMC_np[1], 'e1': best_HMC_np[2], 'e2': best_HMC_np[3], 'center_x': best_HMC_np[4], 'center_y': best_HMC_np[5]}  #Mass distribution model
    kwargs_shear = {'gamma1': best_HMC_np[6], 'gamma2': best_HMC_np[7]}
    kwargs_lens = [kwargs_spep, kwargs_shear]

#     lens_center_x = truth[0][0]['center_x']
#     lens_center_y = truth[0][0]['center_y']
#     angle = 0.*np.pi/180
#     x_arcsec_modeled, y_arcsec_modeled = ((x_arcsec_modeled-lens_center_x)*jnp.cos(angle) - (y_arcsec_modeled-lens_center_y)*jnp.sin(angle) + lens_center_x,(x_arcsec_modeled-lens_center_x)*jnp.sin(angle) + (y_arcsec_modeled-lens_center_y)*jnp.cos(angle) + lens_center_y)

#     kwargs_spep = truth[0][0]
#     kwargs_shear = truth[0][1]
#     kwargs_lens = [kwargs_spep, kwargs_shear]

    sourcePos_x = truth[2][0]['center_x']
    sourcePos_y = truth[2][0]['center_y']


    plt.imshow(modeled_img, norm=mpl.colors.PowerNorm(0.5, vmin=0, vmax=8),extent=extent,)

    lens_plot.lens_model_plot(
    ax,
    lensModel,
    kwargs_lens,
    numPix=num_pix,
    deltaPix=delta_pix,
    point_source=False,
    with_caustics=True,
    with_convergence=False,
    coord_inverse=False,
    fast_caustic=True,
    kwargs_caustics = kwargs_both,
    )


    # modeled_TD = time_delay(x_arcsec_sorted,y_arcsec_sorted,modeled_truth,sourcePos_x_modeled,sourcePos_y_modeled)
    modeled_TD = time_delay(x_arcsec_modeled, y_arcsec_modeled, modeled_truth, sourcePos_x_modeled, sourcePos_y_modeled, z_lens, z_source)
    sort_order_modeled = jnp.argsort(modeled_TD,-1)
    modeled_TD = jnp.take(modeled_TD, sort_order_modeled)
    modeled_TD -= modeled_TD[0]
    time_delay_measured = sim_time_delay_sorted

    modeled_flux = modeled_truth[0][2]['amp']/magnification(x_arcsec_modeled,y_arcsec_modeled,modeled_truth)
    modeled_flux = jnp.take(modeled_flux, sort_order_modeled)
    # modeled_flux = modeled_truth[0][2]['amp']/magnification(x_arcsec_sorted,y_arcsec_sorted,modeled_truth)
    flux_measured = truth[0][2]['amp']/magnification(x_arcsec_sorted,y_arcsec_sorted,truth)


    for i in np.arange(len(x_arcsec_sorted)):
        annotation_str = (
        # f"{i+1}\n"
        f"Simulated Mag.: {flux_measured[i][0]:.3f}\n"
        f"Predicted Mag.: {modeled_flux[i]:.3f}±{flux_unc[i]:.3f}\n"
        f"Simulated TD: {time_delay_measured[i]:.3f}\n"
        f"Predicted TD: {modeled_TD[i]:.3f}±{TD_unc[i]:.3f}"
        )


        plt.annotate(
        annotation_str,
        (x_arcsec_sorted[i] - np.sign(x_arcsec_sorted[i])*0.4 -0.06, y_arcsec_sorted[i] - 0.02),
        fontsize=10,
        color='white'
        )


        plt.plot(x_arcsec_sorted[i],y_arcsec_sorted[i],'kd', ms = 15, alpha = 0.8, markerfacecolor='none')
        # plt.annotate(f"mod_TD: {np.round(modeled_TD[i], 3)}\nobs_TD: {np.round(time_delay_measured[i], 3)}\nmod_flux: {np.round(modeled_flux[i]/1.5, 3)}\nobs_flux: {np.round(flux_measured[i]/1.5, 3)}",
        #            (x_arcsec_sorted[i]-0.02, y_arcsec_sorted[i]+0.05), fontsize=10, color = 'black')

    plt.plot(x_arcsec_sorted,y_arcsec_sorted,'d', ms = 15, alpha = 0.8, label = 'Observed position',color = 'lightblue')
    plt.plot(sourcePos_x,sourcePos_y,'*', ms = 10, alpha = 0.8, label = 'True source Position', color = 'yellow')


plt.plot(x_arcsec_modeled,y_arcsec_modeled,'d', ms = 10, alpha = 0.8, label = 'Predicted position', color = 'navy')
# plt.gca().invert_yaxis()
# plt.gca().invert_xaxis()
plt.grid(False)
plt.xticks([])
plt.yticks([])


scale_length = 0.1/(num_pix*delta_pix)
plt.axhline(xmin=(-0.2 + num_pix*delta_pix/2)/(num_pix*delta_pix), xmax=(-0.2 + num_pix*delta_pix/2)/(num_pix*delta_pix) + scale_length, y=.20, color='white', linewidth=2) #xmin,xmax in data units (Given in proportion of the image)
plt.text(-.20 + 0.1/2 , 0.20-0.01, '0.1"', ha='center', va='bottom', color = 'white')

plt.tight_layout()
plt.legend(loc = 'upper right', facecolor = 'lightgray', labelspacing = 1.)
plt.show()

### Residuals

In [ ]:
from lenstronomy.LightModel.light_model import LightModel
from lenstronomy.SimulationAPI.sim_api import SimAPI
from lenstronomy.Data.imaging_data import ImageData
from lenstronomy.Data.pixel_grid import PixelGrid
from lenstronomy.PointSource.point_source import PointSource
from lenstronomy.ImSim.image_model import ImageModel

from lenstronomy.Util import image_util, data_util, util
import lenstronomy.Plots.plot_util as plot_util
from lenstronomy.Data.psf import PSF


##########LENSTRONOMY
def sim_and_resid(modeled_truth,physical_samples,kernel,x_image, y_image, delta_pix, num_pix,background_rms,exp_time, z_lens,z_source, init_amplitude = 1):

    pixels = num_pix
    pixel_size = delta_pix


    # 1. Setup lens model parameters
    lens_model = LensModel(['EPL','SHEAR'], z_lens=z_lens, z_source=z_source)

    if physical_samples is not None:
        best_HMC_np = np.median(physical_samples.reshape((n_params,-1)).T, axis = 0)
        lens_model_kwargs = [
            {'theta_E': best_HMC_np[0], 'gamma': best_HMC_np[1], 'e1': best_HMC_np[2], 'e2': best_HMC_np[3], 'center_x': best_HMC_np[4], 'center_y': best_HMC_np[5]}, #Mass distribution model
            {'gamma1': best_HMC_np[6], 'gamma2': best_HMC_np[7]}  # SHEAR model
        ]
    else:
        lens_model_kwargs = [
            modeled_truth[0][0], modeled_truth[0][1]
        ]

    # 2. Setup point source model (for the supernova)
    if physical_samples is not None:
        magnifs = np.reshape(best_HMC_np[8]/magnification(x_image,y_image,modeled_truth),(4,))
        point_source_model = PointSource(point_source_type_list = ['LENSED_POSITION'],lens_model = lens_model,fixed_magnification_list = [False])
        point_source_kwargs = [{'ra_image': x_image, 'dec_image': y_image, 'point_amp': init_amplitude*best_HMC_np[8]*np.abs(magnifs)}]
    else:
        magnifs = np.reshape(modeled_truth[0][2]['amp']/magnification(x_image,y_image,modeled_truth),(4,))
        point_source_model = PointSource(point_source_type_list = ['LENSED_POSITION'],lens_model = lens_model,fixed_magnification_list = [False])
        point_source_kwargs = [{'ra_image': x_image, 'dec_image': y_image, 'point_amp': init_amplitude*modeled_truth[0][2]['amp']*np.abs(magnifs)}]
        # amp*12.6 for snz

    # 3. Setup the light model for the host galaxy
    # lens_light_model = LightModel(['SERSIC_ELLIPSE']) #IF LENS LIGHT WANTED
    # kwargs_lens_light = [{'amp':1, 'R_sersic': 1, 'n_sersic': n_sersic, 'e1':e1, 'e2':e2, 'center_x': center_x, 'center_y': center_y}]


    # 4. Create imaging data class
    # kwargs_psf = {'psf_type': 'GAUSSIAN', 'fwhm': 0.07, 'pixel_size': pixel_size, 'truncation': 3}
    kwargs_psf = {'psf_type': 'PIXEL', "kernel_point_source": kernel}
    psf_class = PSF(**kwargs_psf)

    kwargs_data = {'background_rms': background_rms, 'exposure_time': exp_time,
               'ra_at_xy_0': -pixels/2*pixel_size, 'dec_at_xy_0': -pixels/2*pixel_size,
               'transform_pix2angle':np.array([[pixel_size, 0.  ], [0.  , pixel_size]]),
               'image_data': np.zeros((pixels, pixels))}
    data_class = ImageData(**kwargs_data)

    # 5. Simulate image
    image_model = ImageModel(data_class=data_class,
                             psf_class=psf_class,
                             lens_model_class=lens_model,
                             # lens_light_model_class = lens_light_model,
                             # source_model_class = source_light_model,
                             point_source_class=point_source_model,
                            )

    # Get the simulated image
    image = image_model.image(kwargs_lens=lens_model_kwargs,
                              # kwargs_lens_light = kwargs_lens_light,
                              kwargs_ps=point_source_kwargs,
                              # kwargs_source=kwargs_source,
                              # lens_light_add=True, #IF LENS LIGHT WANTED
                              # source_add = True,
                              # point_source_add = True,
                             )

    if physical_samples is None:
        poisson = image_util.add_poisson(image, exp_time=exp_time) *0.05
        bkg = image_util.add_background(image, sigma_bkd=background_rms) *0.05
        image = image + poisson + bkg

    return image



def create_circular_mask(h, w, center=None, radius=None):
    '''
    Create circular mask
    '''

    if center is None: # use the middle of the image
        center = (int(w/2), int(h/2))
    if radius is None: # use the smallest distance between the center and image walls
        radius = min(center[0], center[1], w-center[0], h-center[1])

    Y, X = np.ogrid[:h, :w]
    dist_from_center = np.sqrt((X - center[0])**2 + (Y-center[1])**2)

    mask = dist_from_center <= radius
    return mask

In [ ]:
background_rms475 = 0.00519
t_exp475 =  378.

background_rms625 = 0.006907
t_exp625 =  291.0

background_rms814 = 0.00590
t_exp814 = 312.0

background_rms = np.sqrt(background_rms475**2 + background_rms625**2 +background_rms814**2)
t_exp = t_exp475 + t_exp625 + t_exp814
exp_time = t_exp

init_amplitude = 1 #For model
deltaPix = 0.03*4
numPix = round(num_pix*delta_pix/deltaPix*1.5)
observed_img = sim_and_resid(truth,None,kernel,x_arcsec, y_arcsec, deltaPix, numPix,background_rms,exp_time, z_lens,z_source, init_amplitude = 10*init_amplitude)

In [ ]:
x_image = x_arcsec_modeled
y_image = y_arcsec_modeled

modeled_img = sim_and_resid(modeled_truth,physical_samples,kernel,x_image, y_image, deltaPix, numPix,background_rms,exp_time, z_lens,z_source, init_amplitude = 10*init_amplitude)

title_obs = 'Simulated Image'

err_map = np.sqrt(background_rms**2 + observed_img/exp_time)
dof = np.sum(np.where(observed_img,1,0)) - n_params
residual = (observed_img - modeled_img) #data - model
red_chi_sq = np.sum((residual/err_map)**2)/dof #Compute chi-square in the region of the image where the system is

extent = (-numPix/2*deltaPix - deltaPix/2, numPix/2*deltaPix - deltaPix/2, -numPix/2*deltaPix - deltaPix/2, numPix/2*deltaPix - deltaPix/2)


from mpl_toolkits.axes_grid1 import make_axes_locatable
# fig,ax = plt.subplots(1,3,figsize = (30,30), sharex=False, sharey=False)
fig,(ax0,ax1) = plt.subplots(1,2,figsize = (15,7), sharex=False, sharey=False)
# fig,ax1 = plt.subplots(1,1,figsize = (7.5,7), sharex=False, sharey=False)





scale_length = 1 #arcsec
x_init = 0.08 #Proportion of image
x_init_arcsec = (x_init - 1/2)*(numPix*deltaPix) #arcsec
y_init = 0.05 #Proportion of image
y_init_arcsec = (y_init - 1/2)*(numPix*deltaPix) #arcsec


obs_ax = ax0.imshow(observed_img,
                      extent=extent,
                      cmap = 'inferno',
                      norm=mpl.colors.PowerNorm(0.65,
                                                vmin=0, vmax=1.75
                                                ),
                      origin = 'lower')
ax0.axhline(xmin=x_init, xmax=x_init + scale_length/(numPix*deltaPix), y = y_init_arcsec, color='white', linewidth=2) #xmin,xmax in data units (Given in proportion of the image)
ax0.text(x_init_arcsec + scale_length*0.5, y_init_arcsec + 0.01*(numPix*deltaPix), f'{scale_length}"', ha='center', va='bottom', color = 'white')
ax0.set_title(title_obs)
# ax0.axis('off')


model_ax = ax1.imshow(modeled_img,
                        extent=extent,
                        cmap = 'inferno',
                        norm=mpl.colors.PowerNorm(0.65,
                                                  vmin=0, vmax=1.75
                                                  ),
                        origin = 'lower')
# lens_plot.caustics_plot(ax1, _coords, lensModel, kwargs_lens, fast_caustic=True, color_crit='red', color_caustic=(0,0,0,0),)
ax1.axhline(xmin=x_init, xmax=x_init + scale_length/(numPix*deltaPix), y = y_init_arcsec, color='white', linewidth=2) #xmin,xmax in data units (Given in proportion of the image)
ax1.text(x_init_arcsec + scale_length*0.5, y_init_arcsec + 0.01*(numPix*deltaPix), f'{scale_length}"', ha='center', va='bottom', color = 'white')
# ax1.set_xlim([-numPix/2*deltaPix, numPix/2*deltaPix])
# ax1.set_ylim([-numPix/2*deltaPix, numPix/2*deltaPix])
ax1.set_title('Best-fit Model')
# ax1.axis('off')



# residual_ax = ax[2].imshow((residual/err_map),
#                            extent=extent,
#                            cmap='coolwarm', interpolation='none',
#                            # vmin = -5.5, vmax = 5.5,
#                            origin = 'lower')
# ax[2].text(x_init_arcsec, y_init_arcsec + 0.8*(numPix*deltaPix), '$\chi^2/DOF$ = '+str(round(red_chi_sq,2)), fontsize=12, ha='left', va='bottom')
# ax[2].axhline(xmin=x_init, xmax=x_init + scale_length/(numPix*deltaPix), y = y_init_arcsec, color='black', linewidth=2) #xmin,xmax in data units (Given in proportion of the image)
# ax[2].text(x_init_arcsec + scale_length*0.6, y_init_arcsec + 0.02*(numPix*deltaPix), f'{scale_length}"', ha='center', va='bottom', color = 'black')
# ax[2].set_title('(Observed$-$Model)/$\sigma$')
# # ax[2].axis('off')



divider = make_axes_locatable(ax0)
cax = divider.append_axes('right', size='5%', pad=0.05)
colorbar = fig.colorbar(obs_ax, extend = 'max', cax=cax, orientation='vertical')
colorbar.outline.set_edgecolor('grey')
colorbar.outline.set_linewidth(0.5)


divider = make_axes_locatable(ax1)
cax = divider.append_axes('right', size='5%', pad=0.05)
colorbar = fig.colorbar(model_ax, extend = 'max', cax=cax, orientation='vertical')
colorbar.outline.set_edgecolor('grey')
colorbar.outline.set_linewidth(0.5)


# divider = make_axes_locatable(ax[2])
# cax = divider.append_axes('right', size='5%', pad=0.05)
# colorbar = fig.colorbar(residual_ax, cax=cax, orientation='vertical')
# colorbar.outline.set_edgecolor('grey')
# colorbar.outline.set_linewidth(0.5)


lens_model_list = ['EPL', 'SHEAR']
lensModel = LensModel(lens_model_list=lens_model_list)

best_HMC_np = np.median(physical_samples.reshape((n_params,-1)).T, axis = 0)
kwargs_lens = [
    {'theta_E': best_HMC_np[0], 'gamma': best_HMC_np[1], 'e1': best_HMC_np[2], 'e2': best_HMC_np[3], 'center_x': best_HMC_np[4], 'center_y': best_HMC_np[5]}, #Mass distribution model
    {'gamma1': best_HMC_np[6], 'gamma2': best_HMC_np[7]}]  # SHEAR model

if use_measurements:
    lens_plot.lens_model_plot(
    ax1,
    lensModel,
    kwargs_lens,
    numPix=numPix*10,
    deltaPix=deltaPix/10,
    point_source=False,
    with_caustics=True,
    with_convergence=False,
    coord_inverse=False,
    fast_caustic=True,
    kwargs_caustics = kwargs_both,
    coord_center_ra = - deltaPix/2,
    coord_center_dec = - deltaPix/2,
    )


    for i in np.arange(len(x_arcsec_measured)):


        ## FORMATTING A REAL SYSTEM WITH ASYMM UNCS
        annotation_str = (
        f"Observed Mag.: {flux_measured[i]:.2f}$^{{+{flux_measured_unc_plus[i]:.2f}}}_{{-{flux_measured_unc_minus[i]:.2f}}}$\n"
        f"Predicted Mag.: {modeled_flux[i]:.2f}$\pm${flux_unc[i]:.2f}\n"
        f"Observed TD: {time_delay_measured[i]:.2f}$^{{+{time_delay_measured_unc_plus[i]:.2f}}}_{{-{time_delay_measured_unc_minus[i]:.2f}}}$\n"
        f"Predicted TD: {modeled_TD[i]:.2f}$\pm${TD_unc[i]:.2f}"
        )
        ##



        # annotation_str = (
        # # f"{i+1}\n"
        # f"Observed mag.: {flux_measured[i]:.2f}\n"
        # f"Predicted mag.: {modeled_flux[i]:.2f}$\pm${flux_unc[i]:.2f}\n"
        # f"Observed TD: {time_delay_measured[i]:.2f}\n"
        # f"Predicted TD: {modeled_TD[i]:.2f}$\pm${TD_unc[i]:.2f}"
        # )

        if i == 0:
          ad_hoc_pos_x = -0.07*numPix*deltaPix
          ad_hoc_pos_y = 0.05*numPix*deltaPix
          # ad_hoc_pos = 0.0
        elif i ==1:
          ad_hoc_pos_x = 0.04*numPix*deltaPix
          ad_hoc_pos_y = 0.04*numPix*deltaPix
        else:
          ad_hoc_pos_x = 0.0
          ad_hoc_pos_y = 0.0


        ax1.annotate(
        annotation_str,
        (x_arcsec_measured[i] + np.sign(x_arcsec_measured[i])*numPix*deltaPix*0.08 - 0.22*numPix*deltaPix + np.sign(y_arcsec_measured[i])*np.sign(x_arcsec_measured[i])*numPix*deltaPix*0.0 + ad_hoc_pos_x,
         y_arcsec_measured[i] - np.sign(y_arcsec_measured[i])*numPix*deltaPix*0.0 - 0.2*numPix*deltaPix - np.sign(x_arcsec_measured[i])*numPix*deltaPix*0.0 + ad_hoc_pos_y),
        fontsize=14,
        color='white'
        )

        letter_image = ['A','B','D','C']
        ax1.annotate(
        letter_image[i],
        (x_arcsec_measured[i] - numPix*deltaPix*0.11, y_arcsec_measured[i] + numPix*deltaPix*0.07),
        fontsize=25,
        color='white'
        )

        ax1.plot(x_arcsec_measured[i],y_arcsec_measured[i],'kd', ms = 15, alpha = 0.8, markerfacecolor='none')

    ax1.plot(x_arcsec_measured,y_arcsec_measured,'d', ms = 15, alpha = 0.8, label = 'Observed position',color = 'lightblue')
    ax1.plot(sourcePos_x_modeled,sourcePos_y_modeled,'*', ms = 10, alpha = 0.8, label = 'Predicted source position', color = 'yellow')

else:
    sourcePos_x_truth = truth[2][0]['center_x']
    sourcePos_y_truth = truth[2][0]['center_y']



    lens_plot.lens_model_plot(
    ax1,
    lensModel,
    kwargs_lens,
    numPix=numPix*10,
    deltaPix=deltaPix/10,
    point_source=False,
    with_caustics=True,
    with_convergence=False,
    coord_inverse=False,
    fast_caustic=True,
    kwargs_caustics = kwargs_both,
    coord_center_ra = - deltaPix/2,
    coord_center_dec = - deltaPix/2,
    )

    ax1.set_xlabel('')
    ax1.set_ylabel('')


    for i in np.arange(len(x_arcsec_sorted)):
        annotation_str = (
        # f"{i+1}\n"
        f"Simulated Mag.: {flux_measured[i][0]:.3f}\n"
        f"Predicted Mag.: {modeled_flux[i]:.3f}$\pm${flux_unc[i]:.3f}\n"
        f"Simulated TD: {time_delay_measured[i]:.3f}\n"
        f"Predicted TD: {modeled_TD[i]:.3f}$\pm${TD_unc[i]:.3f}"
        )
        if i == 0:
          ad_hoc_pos = -0.05*numPix*deltaPix
          # ad_hoc_pos = 0.0
        elif i ==2:
          ad_hoc_pos = -0.05*numPix*deltaPix
        else:
          ad_hoc_pos = 0.0

        ax1.annotate(
        annotation_str,
        (x_arcsec_sorted[i] + np.sign(x_arcsec_sorted[i])*numPix*deltaPix*0.1 -0.2*numPix*deltaPix + np.sign(y_arcsec_sorted[i])*np.sign(x_arcsec_sorted[i])*numPix*deltaPix*0.0 + ad_hoc_pos,
         y_arcsec_sorted[i] - np.sign(y_arcsec_sorted[i])*numPix*deltaPix*0.0 - 0.17*numPix*deltaPix - np.sign(x_arcsec_sorted[i])*numPix*deltaPix*0.0),
        fontsize=14,
        color='white'
        )

    ax1.plot(x_arcsec_sorted,y_arcsec_sorted,'d', ms = 15, alpha = 0.8, label = 'Observed position',color = 'lightblue')
    ax1.plot(sourcePos_x_truth,sourcePos_y_truth,'*', ms = 10, alpha = 0.8, label = 'True source position', color = 'yellow')

if use_measurements:
    # Add compass annotations
    # Define compass directions and offsets
    directions = ['N', 'E']
    positions = [
        (compass_x_center, compass_y_center),
        (compass_x_center, compass_y_center),
    ]

    for direction, (pos_x, pos_y), (dx, dy) in zip(directions, positions, offsets):
        ax1.text(pos_x + 1.8*dx, pos_y + 1.8*dy, direction, ha='center', va='center', fontsize=18,
                fontweight=10,
                color='white')
        ax1.arrow(pos_x, pos_y, dx, dy, color='white',
                  width = 0.00001,
                  head_width=-0.015 * (extent[1] - extent[0]),
                  head_length=-0.03 * (extent[2] - extent[3]),
                )


ax1.plot(x_arcsec_modeled,y_arcsec_modeled,'d', ms = 10, alpha = 0.8, label = 'Predicted position', color = 'navy')

plt.tight_layout()
handles, labels = ax1.get_legend_handles_labels()


keep = [
    (h, l)
    for h, l in zip(handles, labels)
    if l not in ("caustics", "critical curves", "source position")
]

handles, labels = zip(*keep)

if use_measurements:
    order = [0, 2, 1]

else:
    order = [2, 0, 1]

ax1.legend(handles = [handles[i] for i in order], labels = [labels[i] for i in order], loc = 'upper right', facecolor = 'lightgray', labelspacing = 1., fontsize = 12)
plt.show()


print('Total Modeling time ',timedelta(seconds=elapsed_modeling))
print('worse rhat:', rhat[worst_idx])
print('Max and min ESS', ess.max(), ess.min())